In [ ]:
# Cell 1 - imports
import json
import os
from pathlib import Path
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np
from IPython.display import Image as DisplayImage, Video, display
from PIL import Image, ImageDraw

In [ ]:
# Display and detection helpers
VIDEO_DIR = Path("generated_videos")
VIDEO_DIR.mkdir(exist_ok=True)
def save_video(frames, path, fps=20):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frames = frames.astype(np.uint8)
    try:
        imageio.mimsave(path, frames, fps=fps)
        return path
    except Exception as exc:
        fallback = path.with_suffix(".gif")
        print(f"MP4 save skipped: {type(exc).__name__}: {exc}")
        print(f"Saving GIF fallback: {fallback}")
        imageio.mimsave(fallback, frames, duration=1 / fps)
        return fallback
def show_video(frames, name, fps=20, embed=True):
    path = save_video(frames, VIDEO_DIR / name, fps=fps)
    if path.suffix.lower() == ".gif":
        display(DisplayImage(filename=str(path)))
    else:
        display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
    return path
def show_video_file(path, embed=True):
    path = Path(path)
    display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
    return path
def clip_indices(num_frames, clip_frames=48, seed=0):
    if num_frames <= 0:
        return np.asarray([], dtype=int)
    clip_frames = min(int(clip_frames), int(num_frames))
    rng = np.random.default_rng(seed)
    max_start = max(0, int(num_frames) - clip_frames)
    start = int(rng.integers(0, max_start + 1)) if max_start else 0
    return np.arange(start, start + clip_frames, dtype=int)
def bbox_from_mask(mask, min_area=50):
    mask = mask.astype(bool)
    visited = np.zeros(mask.shape, dtype=bool)
    boxes = []
    height, width = mask.shape
    ys, xs = np.nonzero(mask)
    for y0, x0 in zip(ys, xs):
        if visited[y0, x0] or not mask[y0, x0]:
            continue
        stack = [(int(y0), int(x0))]
        visited[y0, x0] = True
        x_min = x_max = int(x0)
        y_min = y_max = int(y0)
        area = 0
        while stack:
            y, x = stack.pop()
            area += 1
            x_min = min(x_min, x)
            x_max = max(x_max, x)
            y_min = min(y_min, y)
            y_max = max(y_max, y)
            for ny in (y - 1, y, y + 1):
                for nx in (x - 1, x, x + 1):
                    if ny == y and nx == x:
                        continue
                    if 0 <= ny < height and 0 <= nx < width and mask[ny, nx] and not visited[ny, nx]:
                        visited[ny, nx] = True
                        stack.append((ny, nx))
        if area >= min_area:
            boxes.append((x_min, y_min, x_max + 1, y_max + 1, float(area)))
    return boxes
def draw_boxes(ax, boxes, color="lime", labels=None):
    for i, box in enumerate(boxes):
        x1, y1, x2, y2 = box[:4]
        rect = patches.Rectangle((x1, y1), x2 - x1, y2 - y1, fill=False, linewidth=2, edgecolor=color)
        ax.add_patch(rect)
        if labels:
            ax.text(x1, max(0, y1 - 4), labels[i], color=color, fontsize=9, weight="bold")
def show_detection_frame(frames, frame_idx, boxes, title, labels=None):
    plt.figure(figsize=(8, 5))
    ax = plt.gca()
    ax.imshow(frames[frame_idx])
    draw_boxes(ax, boxes, labels=labels)
    ax.set_title(title)
    ax.axis("off")
    plt.show()
def plot_detection_metric(values, title, ylabel="objects"):
    plt.figure(figsize=(8, 3))
    plt.plot(values)
    plt.title(title)
    plt.xlabel("frame")
    plt.ylabel(ylabel)
    plt.grid(alpha=0.25)
    plt.show()


In [ ]:
# Internet video helpers for independent visual problems 4 and 5
def download_video(url, path):
    import urllib.request
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    if not path.exists():
        print(f"Downloading {url}")
        urllib.request.urlretrieve(url, path)
    return path
def read_video(path, max_frames=None, stride=1, start_frame=0):
    import cv2
    cap = cv2.VideoCapture(str(path))
    if start_frame:
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))
    frames_out = []
    frame_no = 0
    while True:
        ok, frame = cap.read()
        if not ok:
            break
        if frame_no % stride == 0:
            frames_out.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            if max_frames is not None and len(frames_out) >= max_frames:
                break
        frame_no += 1
    cap.release()
    if not frames_out:
        raise RuntimeError(f"No frames read from {path}")
    return np.stack(frames_out)
def read_video_sample(path, max_frames=120, stride=3, start_seconds=0):
    import cv2
    cap = cv2.VideoCapture(str(path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    cap.release()
    return read_video(path, max_frames=max_frames, stride=stride, start_frame=int(start_seconds * fps))


# Visual Problems


## 1 - Lunar Lander


In [ ]:
# LunarLander setup - use kernel: Python (Action Inference)
import sys
import gymnasium as gym
from gymnasium.envs.box2d.lunar_lander import heuristic
ACTION_NAMES = {
    0: "noop",
    1: "left_engine",
    2: "main_engine",
    3: "right_engine",
}
print(sys.executable)
print(gym.__version__)


In [ ]:
# LunarLander rollout
def rollout_lunar_lander(policy, seed=0, max_steps=300, render_mode="rgb_array"):
    env = gym.make("LunarLander-v3", render_mode=render_mode)
    obs, info = env.reset(seed=seed)
    observations, actions, rewards, frames, infos = [], [], [], [], []
    for t in range(max_steps):
        action = int(policy(env.unwrapped, obs))
        next_obs, reward, terminated, truncated, info = env.step(action)
        observations.append(obs.copy())
        actions.append(action)
        rewards.append(float(reward))
        infos.append(info)
        frames.append(env.render())
        obs = next_obs
        if terminated or truncated:
            break
    env.close()
    return {
        "observations": np.asarray(observations, dtype=np.float32),
        "actions": np.asarray(actions, dtype=np.int64),
        "rewards": np.asarray(rewards, dtype=np.float32),
        "frames": np.stack(frames),
        "infos": infos,
    }
rollout = rollout_lunar_lander(heuristic, seed=0)
frames = rollout["frames"]
actions = rollout["actions"]
observations = rollout["observations"]
print("frames:", frames.shape)
print("observations:", observations.shape)
print("actions:", actions.shape)
print("action counts:", {ACTION_NAMES[i]: int((actions == i).sum()) for i in ACTION_NAMES})


In [ ]:
frame_idx = min(10, len(frames) - 1)
plt.figure(figsize=(7, 5))
plt.imshow(frames[frame_idx])
plt.title(f"t={frame_idx}, action={actions[frame_idx]} ({ACTION_NAMES[int(actions[frame_idx])]})")
plt.axis("off")
plt.show()


In [ ]:
show_video(frames, "1_lunar_lander.mp4", fps=30)


## 2 - Car Racing


In [ ]:
# Install once if Box2D is missing
!pip install gymnasium[box2d] -q


In [ ]:
# CarRacing with downloaded solved PPO replay, loaded at full downloaded length.
from pathlib import Path
import imageio.v2 as imageio
import numpy as np

CAR_RACING_SOLVED_VIDEO_PATH = Path("artifacts/raw_videos/2_car_racing_solved_ppo_replay.mp4")
if not CAR_RACING_SOLVED_VIDEO_PATH.exists():
    raise FileNotFoundError(f"Missing solved CarRacing replay: {CAR_RACING_SOLVED_VIDEO_PATH}")

def load_rgb_video_all(path):
    reader = imageio.get_reader(path)
    try:
        frames_out = [np.asarray(frame)[..., :3] for frame in reader]
    finally:
        reader.close()
    if not frames_out:
        raise RuntimeError(f"No frames loaded from {path}")
    return np.stack(frames_out).astype(np.uint8)

frames_car = load_rgb_video_all(CAR_RACING_SOLVED_VIDEO_PATH)
car_frame_idx = min(40, len(frames_car) - 1)
print("loaded solved CarRacing frames:", frames_car.shape)
frames_car.shape


In [ ]:
car_frame_idx = min(40, len(frames_car) - 1)
plt.figure(figsize=(7, 5))
plt.imshow(frames_car[car_frame_idx])
plt.title(f"CarRacing frame {car_frame_idx}")
plt.axis("off")
plt.show()


In [ ]:
show_video(frames_car, "2_car_racing_solved_full.mp4", fps=30, embed=False)


## 3 - Recorded Traffic With People


In [ ]:
# Load recorded traffic video
import cv2
import urllib.request
if not Path("artifacts/raw_videos/traffic.avi").exists():
    url = "https://raw.githubusercontent.com/opencv/opencv_extra/master/testdata/cv/video/768x576.avi"
    urllib.request.urlretrieve(url, "artifacts/raw_videos/traffic.avi")
cap = cv2.VideoCapture("artifacts/raw_videos/traffic.avi")
frames_medium = []
while True:
    ok, frame = cap.read()
    if not ok:
        break
    frames_medium.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
cap.release()
frames_medium = np.stack(frames_medium)
frames_medium.shape


In [ ]:
traffic_frame_idx = min(1, len(frames_medium) - 1)
plt.figure(figsize=(8, 5))
plt.imshow(frames_medium[traffic_frame_idx])
plt.title(f"Recorded traffic frame {traffic_frame_idx}")
plt.axis("off")
plt.show()


In [ ]:
show_video(frames_medium, "3_recorded_traffic_people.mp4", fps=20)


## 4 - Random YouTube Driving Scene


In [ ]:
# Problem 4: random 3-minute YouTube driving clip
PROBLEM_4_URL = "https://www.youtube.com/watch?v=7EovwWQIvBo"
problem_4_path = Path("artifacts/raw_videos/problem_4_youtube_random.mp4")
if not problem_4_path.exists():
    raise FileNotFoundError(f"Missing {problem_4_path}. Download it with yt-dlp first.")
# Display the full downloaded 3-minute clip.
show_video_file(problem_4_path, embed=False)
# Use a small cached sample for plotting/models. Do not decode the full 3-minute file every run.
if "frames_problem4" not in globals():
    frames_problem4 = read_video_sample(problem_4_path, max_frames=120, stride=3, start_seconds=20)
problem4_frame_idx = min(30, len(frames_problem4) - 1)
plt.figure(figsize=(8, 5))
plt.imshow(frames_problem4[problem4_frame_idx])
plt.title(f"Problem 4 sampled frame {problem4_frame_idx}")
plt.axis("off")
plt.show()


## 5 - Hard Vehicle-Crowd Interaction


In [ ]:
# Problem 5: hardest 3-minute vehicle-crowd YouTube clip
PROBLEM_5_URL = "https://www.youtube.com/watch?v=7HaJArMDKgI"
problem_5_path = Path("artifacts/raw_videos/problem_5_youtube_hardest.mp4")
if not problem_5_path.exists():
    raise FileNotFoundError(f"Missing {problem_5_path}. Download it with yt-dlp first.")
# Display the full downloaded 3-minute clip.
show_video_file(problem_5_path, embed=False)
# Use a small cached sample for plotting/models. Do not decode the full 3-minute file every run.
if "frames_problem5" not in globals():
    frames_problem5 = read_video_sample(problem_5_path, max_frames=120, stride=3, start_seconds=20)
problem5_frame_idx = min(40, len(frames_problem5) - 1)
plt.figure(figsize=(8, 5))
plt.imshow(frames_problem5[problem5_frame_idx])
plt.title(f"Problem 5 sampled frame {problem5_frame_idx}")
plt.axis("off")
plt.show()


# Object Detection


In [ ]:
# YOLO setup and display utilities. Run this once before the detection subsections.
!pip install ultralytics -q
from ultralytics import YOLO
import pandas as pd
model = YOLO("yolo11n.pt")
def run_yolo(frames_batch):
    results = model(list(frames_batch), stream=True, verbose=False)
    all_detections = []
    for r in results:
        boxes = r.boxes
        frame_dets = [
            (int(c), float(conf), *map(float, xyxy))
            for c, conf, xyxy in zip(boxes.cls, boxes.conf, boxes.xyxy)
        ]
        all_detections.append(frame_dets)
    return all_detections
def yolo_dets(frame_dets, conf_min=0.25):
    boxes, labels = [], []
    for class_id, conf, x1, y1, x2, y2 in frame_dets:
        if conf >= conf_min:
            boxes.append((x1, y1, x2, y2, conf))
            labels.append(f"{model.names[int(class_id)]} {conf:.2f}")
    return boxes, labels
def yolo_detection_counts(all_detections, conf_min=0.25):
    return np.asarray([sum(1 for _, conf, *_ in frame_dets if conf >= conf_min) for frame_dets in all_detections], dtype=int)
def box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = area_a + area_b - inter
    return float(inter / denom) if denom > 0 else 0.0
def track_yolo_detections(detections_batch, conf_min=0.25, iou_threshold=0.3, max_missing=2):
    tracked_frames = []
    active_tracks = []
    next_track_id = 0
    for frame_no, frame_dets in enumerate(detections_batch):
        old_track_count = len(active_tracks)
        detections = []
        for class_id, conf, x1, y1, x2, y2 in frame_dets:
            if conf < conf_min:
                continue
            detections.append({
                "frame": frame_no,
                "class_id": int(class_id),
                "class_name": model.names[int(class_id)],
                "confidence": float(conf),
                "box": (float(x1), float(y1), float(x2), float(y2)),
            })
        matches = []
        used_tracks = set()
        used_dets = set()
        candidates = []
        for track_idx, track in enumerate(active_tracks):
            if track["missing"] > max_missing:
                continue
            for det_idx, det in enumerate(detections):
                if det["class_id"] != track["class_id"]:
                    continue
                iou_score = box_iou(track["box"], det["box"])
                if iou_score >= iou_threshold:
                    candidates.append((iou_score, track_idx, det_idx))
        for _, track_idx, det_idx in sorted(candidates, reverse=True):
            if track_idx in used_tracks or det_idx in used_dets:
                continue
            matches.append((track_idx, det_idx))
            used_tracks.add(track_idx)
            used_dets.add(det_idx)
        frame_records = []
        for track_idx, det_idx in matches:
            track = active_tracks[track_idx]
            det = detections[det_idx]
            det["track_id"] = track["track_id"]
            det["matched_iou"] = box_iou(track["box"], det["box"])
            track.update({"box": det["box"], "class_id": det["class_id"], "missing": 0})
            frame_records.append(det)
        for det_idx, det in enumerate(detections):
            if det_idx in used_dets:
                continue
            det["track_id"] = next_track_id
            det["matched_iou"] = np.nan
            active_tracks.append({"track_id": next_track_id, "class_id": det["class_id"], "box": det["box"], "missing": 0})
            next_track_id += 1
            frame_records.append(det)
        for track_idx, track in enumerate(active_tracks[:old_track_count]):
            if track_idx not in used_tracks:
                track["missing"] += 1
        active_tracks = [track for track in active_tracks if track["missing"] <= max_missing]
        tracked_frames.append(sorted(frame_records, key=lambda obj: obj["track_id"]))
    return tracked_frames
def detection_tracks_dataframe(tracked_frames):
    rows = []
    for frame_records in tracked_frames:
        for obj in frame_records:
            x1, y1, x2, y2 = obj["box"]
            rows.append({
                "frame": obj["frame"],
                "track_id": obj["track_id"],
                "class_name": obj["class_name"],
                "confidence": round(obj["confidence"], 3),
                "x1": round(x1, 2),
                "y1": round(y1, 2),
                "x2": round(x2, 2),
                "y2": round(y2, 2),
                "width": round(x2 - x1, 2),
                "height": round(y2 - y1, 2),
                "matched_iou": None if np.isnan(obj["matched_iou"]) else round(obj["matched_iou"], 3),
            })
    return pd.DataFrame(rows)
def show_detection_data(tracked_frames, title):
    df = detection_tracks_dataframe(tracked_frames)
    print(title)
    display(df.head(50))
    return df
def draw_yolo_tracked_frame(frame, frame_records):
    import cv2
    image = frame.copy()
    for obj in frame_records:
        x1, y1, x2, y2 = [int(v) for v in obj["box"]]
        label = f"T{obj['track_id']} {obj['class_name']} {obj['confidence']:.2f}"
        cv2.rectangle(image, (x1, y1), (x2, y2), (0, 255, 0), 2)
        cv2.putText(image, label, (x1, max(15, y1 - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (0, 255, 0), 1, cv2.LINE_AA)
    return image
def draw_yolo_frame(frame, frame_dets, conf_min=0.25):
    boxes, labels = yolo_dets(frame_dets, conf_min=conf_min)
    records = []
    for object_id, (box, label) in enumerate(zip(boxes, labels)):
        class_name, conf = label.rsplit(" ", 1)
        records.append({"track_id": object_id, "class_name": class_name, "confidence": float(conf), "box": box[:4]})
    return draw_yolo_tracked_frame(frame, records)
def show_detection_video(frames_batch, detections_batch, name, fps=8, clip_frames=48, seed=0):
    indices = clip_indices(min(len(frames_batch), len(detections_batch)), clip_frames=clip_frames, seed=seed)
    clip_detections = [detections_batch[i] for i in indices]
    clip_tracks = track_yolo_detections(clip_detections)
    annotated = [draw_yolo_tracked_frame(frames_batch[i], frame_records) for i, frame_records in zip(indices, clip_tracks)]
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    return show_video(np.asarray(annotated), name, fps=fps)


## 1 - Lunar Lander Detection


In [ ]:
# YOLO decides what, if anything, it recognizes in LunarLander. No manual class labels.
if "detections_lunar" not in globals():
    detections_lunar = run_yolo(frames)
if "lunar_yolo_counts" not in globals():
    lunar_yolo_counts = yolo_detection_counts(detections_lunar)
if "lunar_yolo_tracks" not in globals():
    lunar_yolo_tracks = track_yolo_detections(detections_lunar)
boxes, labels = yolo_dets(detections_lunar[frame_idx])
print(labels if labels else "YOLO detected nothing on the selected frame")
show_detection_video(frames, detections_lunar, "1_lunar_lander_yolo_detection.mp4", fps=8, clip_frames=48, seed=0)
plot_detection_metric(lunar_yolo_counts, "1 - YOLO detections per frame", ylabel="detections")
lunar_yolo_data = show_detection_data(lunar_yolo_tracks, "1 - Lunar Lander YOLO tracking data representation")


## 2 - Car Racing Detection


In [ ]:
# YOLO decides what, if anything, it recognizes in CarRacing. No manual class labels.
if "detections_car" not in globals():
    detections_car = run_yolo(frames_car)
if "car_yolo_counts" not in globals():
    car_yolo_counts = yolo_detection_counts(detections_car)
if "car_yolo_tracks" not in globals():
    car_yolo_tracks = track_yolo_detections(detections_car)
boxes, labels = yolo_dets(detections_car[car_frame_idx])
print(labels if labels else "YOLO detected nothing on the selected frame")
show_detection_video(frames_car, detections_car, "2_car_racing_yolo_detection.mp4", fps=8, clip_frames=48, seed=0)
plot_detection_metric(car_yolo_counts, "2 - YOLO detections per frame", ylabel="detections")
car_yolo_data = show_detection_data(car_yolo_tracks, "2 - Car Racing YOLO tracking data representation")


## 3 - Recorded Traffic With People Detection


In [ ]:
if "detections_traffic" not in globals():
    detections_traffic = run_yolo(frames_medium)
if "traffic_yolo_counts" not in globals():
    traffic_yolo_counts = yolo_detection_counts(detections_traffic)
if "traffic_yolo_tracks" not in globals():
    traffic_yolo_tracks = track_yolo_detections(detections_traffic)
det_frame_idx = min(10, len(frames_medium) - 1)
boxes, labels = yolo_dets(detections_traffic[det_frame_idx])
print(labels if labels else "YOLO detected nothing on the selected frame")
show_detection_video(frames_medium, detections_traffic, "3_recorded_traffic_yolo_detection.mp4", fps=8, clip_frames=48, seed=0)
plot_detection_metric(traffic_yolo_counts, "3 - YOLO detections per frame", ylabel="detections")
traffic_yolo_data = show_detection_data(traffic_yolo_tracks, "3 - Recorded Traffic YOLO tracking data representation")


## 4 - Random YouTube Driving Scene Detection


In [ ]:
if "detections_problem4" not in globals():
    detections_problem4 = run_yolo(frames_problem4)
if "problem4_yolo_counts" not in globals():
    problem4_yolo_counts = yolo_detection_counts(detections_problem4)
if "problem4_yolo_tracks" not in globals():
    problem4_yolo_tracks = track_yolo_detections(detections_problem4)
problem4_det_frame_idx = min(30, len(frames_problem4) - 1)
boxes, labels = yolo_dets(detections_problem4[problem4_det_frame_idx])
print(labels if labels else "YOLO detected nothing on the selected frame")
show_detection_video(frames_problem4, detections_problem4, "4_youtube_driving_yolo_detection.mp4", fps=8, clip_frames=48, seed=0)
plot_detection_metric(problem4_yolo_counts, "4 - YOLO detections per sampled frame", ylabel="detections")
problem4_yolo_data = show_detection_data(problem4_yolo_tracks, "4 - Random YouTube Driving YOLO tracking data representation")


## 5 - Hard Vehicle-Crowd Interaction Detection


In [ ]:
if "detections_problem5" not in globals():
    detections_problem5 = run_yolo(frames_problem5)
if "problem5_yolo_counts" not in globals():
    problem5_yolo_counts = yolo_detection_counts(detections_problem5)
if "problem5_yolo_tracks" not in globals():
    problem5_yolo_tracks = track_yolo_detections(detections_problem5)
problem5_det_frame_idx = min(40, len(frames_problem5) - 1)
boxes, labels = yolo_dets(detections_problem5[problem5_det_frame_idx])
print(labels if labels else "YOLO detected nothing on the selected frame")
show_detection_video(frames_problem5, detections_problem5, "5_hard_vehicle_crowd_yolo_detection.mp4", fps=8, clip_frames=48, seed=0)
plot_detection_metric(problem5_yolo_counts, "5 - YOLO detections per sampled frame", ylabel="detections")
problem5_yolo_data = show_detection_data(problem5_yolo_tracks, "5 - Hard Vehicle-Crowd YOLO tracking data representation")


# Object Segmentation


## WINNER - Object Segmentation Blob (Active Agents Only: People + Vehicles)


In [ ]:
# Minimal standalone loader for this section.
from pathlib import Path
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd 
from IPython.display import display
from PIL import Image, ImageDraw
try:
    import cv2
except Exception as exc:
    cv2 = None
    print(f"cv2 unavailable: {type(exc).__name__}: {exc}")
def load_video_frames_minimal(path, max_frames=120, stride=3, start_seconds=0):
    path = Path(path)
    if cv2 is not None:
        cap = cv2.VideoCapture(str(path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_seconds * fps))
        out = []
        frame_no = 0
        while True:
            ok, frame = cap.read()
            if not ok:
                break
            if frame_no % stride == 0:
                out.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                if max_frames is not None and len(out) >= int(max_frames):
                    break
            frame_no += 1
        cap.release()
        if out:
            return np.stack(out)
    reader = imageio.get_reader(path)
    out = []
    for frame_no, frame in enumerate(reader):
        if frame_no % stride == 0:
            out.append(np.asarray(frame)[..., :3])
            if max_frames is not None and len(out) >= int(max_frames):
                break
    reader.close()
    if not out:
        raise RuntimeError(f"No frames loaded from {path}")
    return np.stack(out)
def ensure_frames_var(var_name, path, idx_name, idx_value, max_frames=120, stride=3, start_seconds=0):
    if var_name not in globals():
        globals()[var_name] = load_video_frames_minimal(path, max_frames=max_frames, stride=stride, start_seconds=start_seconds)
        print(f"loaded {var_name}:", globals()[var_name].shape)
    if idx_name not in globals():
        globals()[idx_name] = min(idx_value, len(globals()[var_name]) - 1)
        print(f"set {idx_name}:", globals()[idx_name])
ensure_frames_var("frames", "generated_videos/1_lunar_lander.gif", "frame_idx", 10, max_frames=80, stride=1)
ensure_frames_var("frames_car", "artifacts/raw_videos/2_car_racing_solved_ppo_replay.mp4", "car_frame_idx", 40, max_frames=None, stride=1)
ensure_frames_var("frames_medium", "artifacts/raw_videos/traffic.avi", "traffic_frame_idx", 1, max_frames=120, stride=1)
ensure_frames_var("frames_problem4", "artifacts/raw_videos/problem_4_youtube_random.mp4", "problem4_frame_idx", 30, max_frames=120, stride=3, start_seconds=20)
ensure_frames_var("frames_problem5", "artifacts/raw_videos/problem_5_youtube_hardest.mp4", "problem5_frame_idx", 40, max_frames=120, stride=3, start_seconds=20)
from ultralytics import YOLO
active_agent_seg_model = YOLO("yolov8n-seg.pt")
VIDEO_DIR = Path("generated_videos")
VIDEO_DIR.mkdir(exist_ok=True)
def save_video(frames, path, fps=20):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frames = frames.astype(np.uint8)
    try:
        imageio.mimsave(path, frames, fps=fps)
        return path
    except Exception as exc:
        fallback = path.with_suffix(".gif")
        print(f"MP4 save skipped: {type(exc).__name__}: {exc}")
        print(f"Saving GIF fallback: {fallback}")
        imageio.mimsave(fallback, frames, duration=1 / fps)
        return fallback
def show_video(frames, name, fps=20, embed=True):
    from IPython.display import Image as DisplayImage, Video
    path = save_video(frames, VIDEO_DIR / name, fps=fps)
    if path.suffix.lower() == ".gif":
        display(DisplayImage(filename=str(path)))
    else:
        display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
    return path
def clip_indices(num_frames, clip_frames=48, seed=0):
    if num_frames <= 0:
        return np.asarray([], dtype=int)
    clip_frames = min(int(clip_frames), int(num_frames))
    rng = np.random.default_rng(seed)
    max_start = max(0, int(num_frames) - clip_frames)
    start = int(rng.integers(0, max_start + 1)) if max_start else 0
    return np.arange(start, start + clip_frames, dtype=int)
def mask_to_record(mask, object_id):
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return None
    x1, x2 = int(xs.min()), int(xs.max()) + 1
    y1, y2 = int(ys.min()), int(ys.max()) + 1
    area = int(mask.sum())
    centroid = (float(xs.mean()), float(ys.mean()))
    return {
        "object_id": object_id,
        "mask": mask,
        "box": (x1, y1, x2, y2),
        "area": area,
        "centroid": centroid,
    }
# Active-agent-only segmentation: people + vehicles.
# Background is everything outside these selected class masks.
ACTIVE_AGENT_CLASS_NAMES = {
    "person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat"
}
def active_agent_records_from_yolo_seg(frame, model, conf_min=0.25, max_objects=40):
    result = model(frame, verbose=False, conf=conf_min)[0]
    records = []
    if result.masks is None:
        return records
    masks = result.masks.data.cpu().numpy().astype(bool)
    if masks.shape[1:] != frame.shape[:2]:
        import cv2
        masks = np.asarray([
            cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool)
            for mask in masks
        ])
    classes = result.boxes.cls.cpu().numpy().astype(int)
    confs = result.boxes.conf.cpu().numpy()
    for mask, class_id, conf in zip(masks, classes, confs):
        class_name = result.names[int(class_id)]
        if class_name not in ACTIVE_AGENT_CLASS_NAMES or float(conf) < conf_min:
            continue
        record = mask_to_record(mask, len(records))
        if record is None:
            continue
        record["class_id"] = int(class_id)
        record["class_name"] = class_name
        record["confidence"] = float(conf)
        records.append(record)
    return sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]

def largest_connected_component(mask, min_area=30):
    if cv2 is None:
        return mask.astype(bool)
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    if num_labels <= 1:
        return np.zeros_like(mask, dtype=bool)
    areas = stats[1:, cv2.CC_STAT_AREA]
    best = int(np.argmax(areas)) + 1
    if areas[best - 1] < min_area:
        return np.zeros_like(mask, dtype=bool)
    return labels == best

def lunar_lander_agent_records(frame):
    rgb = frame.astype(np.int16)
    brightness = rgb.mean(axis=2)
    # LunarLander agent/flames are bright colored structures over a dark sky. Keep upper scene, ignore UI floor text.
    colorfulness = np.max(rgb, axis=2) - np.min(rgb, axis=2)
    mask = ((brightness > 65) | (colorfulness > 45)) & (np.indices(frame.shape[:2])[0] < int(frame.shape[0] * 0.86))
    if cv2 is not None:
        mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((5, 5), dtype=np.uint8), iterations=1).astype(bool)
    mask = largest_connected_component(mask, min_area=40)
    record = mask_to_record(mask, 0)
    if record is None:
        return []
    record.update({"class_id": -101, "class_name": "lunar_lander_agent", "confidence": 1.0, "source": "env_specific_color_geometry"})
    return [record]

def car_racing_agent_records(frame):
    rgb = frame.astype(np.int16)
    r, g, b = rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]
    h, w = frame.shape[:2]
    yy, xx = np.indices((h, w))
    # The CarRacing ego car is red and near the lower center of the rendered frame.
    red_car = (r > 120) & (r > g * 1.45) & (r > b * 1.45)
    region = (yy > int(h * 0.38)) & (yy < int(h * 0.84)) & (xx > int(w * 0.25)) & (xx < int(w * 0.75))
    mask = red_car & region
    if cv2 is not None:
        mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((5, 5), dtype=np.uint8), iterations=1).astype(bool)
    mask = largest_connected_component(mask, min_area=25)
    record = mask_to_record(mask, 0)
    if record is None:
        return []
    record.update({"class_id": -102, "class_name": "car_racing_ego_car", "confidence": 1.0, "source": "env_specific_color_geometry"})
    return [record]

def active_agent_records(frame, model=None, conf_min=0.25, max_objects=40, scene_type=None):
    if scene_type == "lunar_lander":
        return lunar_lander_agent_records(frame)
    if scene_type == "car_racing":
        return car_racing_agent_records(frame)
    if model is None:
        model = active_agent_seg_model
    return active_agent_records_from_yolo_seg(frame, model=model, conf_min=conf_min, max_objects=max_objects)

def render_active_agents_inverse_background(frame, records, title, alpha_bg=0.35, alpha_agent=0.55):
    active_mask = np.zeros(frame.shape[:2], dtype=bool)
    for record in records:
        active_mask |= record["mask"]
    bg_mask = ~active_mask
    image = frame.copy()
    image[bg_mask] = (image[bg_mask] * (1 - alpha_bg) + np.array([40, 180, 90]) * alpha_bg).astype(np.uint8)
    image[active_mask] = (image[active_mask] * (1 - alpha_agent) + np.array([230, 50, 40]) * alpha_agent).astype(np.uint8)
    plt.figure(figsize=(8, 5))
    plt.imshow(image)
    plt.title(title)
    plt.axis("off")
    plt.show()
    return bg_mask, active_mask
def show_active_agent_only_segmentation_experiment(frames_batch, frame_idx, title, model=None, conf_min=0.25):
    if model is None:
        model = active_agent_seg_model
    frame = frames_batch[frame_idx]
    records = active_agent_records(frame, model=model, conf_min=conf_min, scene_type=scene_type)
    bg_mask, active_mask = render_active_agents_inverse_background(frame, records, title)
    rows = []
    for obj in records:
        rows.append({
            "object_id": obj["object_id"],
            "class_name": obj["class_name"],
            "confidence": round(obj["confidence"], 3),
            "area": obj["area"],
            "centroid_x": round(obj["centroid"][0], 2),
            "centroid_y": round(obj["centroid"][1], 2),
            "box": obj["box"],
        })
    summary = pd.DataFrame([{
        "frame": int(frame_idx),
        "selected_classes": sorted(ACTIVE_AGENT_CLASS_NAMES),
        "agents": len(records),
        "active_agent_pixels": int(active_mask.sum()),
        "background_pixels": int(bg_mask.sum()),
        "background_ratio": float(bg_mask.mean()),
    }])
    display(summary)
    display(pd.DataFrame(rows))
    return bg_mask, records, summary
def save_active_agent_video(frames_batch, name, model=None, fps=6, clip_frames=36, seed=0, conf_min=0.25, scene_type=None):
    if model is None:
        model = active_agent_seg_model
    Path("generated_videos").mkdir(exist_ok=True)
    indices = np.arange(min(len(frames_batch), clip_frames), dtype=int)
    if len(frames_batch) > clip_frames:
        indices = clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed) if "clip_indices" in globals() else indices
    rendered = []
    raw_sequence = []
    for i in indices:
        records = active_agent_records(frames_batch[i], model=model, conf_min=conf_min, scene_type=scene_type)
        raw_sequence.append(records)
        active_mask = np.zeros(frames_batch[i].shape[:2], dtype=bool)
        for record in records:
            active_mask |= record["mask"]
        bg_mask = ~active_mask
        image = frames_batch[i].copy()
        image[bg_mask] = (image[bg_mask] * 0.65 + np.array([40, 180, 90]) * 0.35).astype(np.uint8)
        image[active_mask] = (image[active_mask] * 0.45 + np.array([230, 50, 40]) * 0.55).astype(np.uint8)
        rendered.append(image)
    tracked_sequence = track_segmentation_sequence(raw_sequence, stable_blob=True) if "track_segmentation_sequence" in globals() else raw_sequence
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    path = show_video(np.asarray(rendered), name, fps=fps) if "show_video" in globals() else None
    return path, tracked_sequence
def show_active_agent_only_segmentation_experiment(frames_batch, frame_idx, title, model=None, conf_min=0.25, video_name=None, scene_type=None):
    if model is None:
        model = active_agent_seg_model
    frame = frames_batch[frame_idx]
    records = active_agent_records(frame, model=model, conf_min=conf_min, scene_type=scene_type)
    bg_mask, active_mask = render_active_agents_inverse_background(frame, records, title)
    video_path = None
    video_tracks = None
    if video_name is not None:
        video_path, video_tracks = save_active_agent_video(frames_batch, video_name, model=model, conf_min=conf_min, scene_type=scene_type)
    rows = []
    for obj in records:
        rows.append({
            "object_id": obj["object_id"],
            "class_name": obj["class_name"],
            "confidence": round(obj["confidence"], 3),
            "area": obj["area"],
            "centroid_x": round(obj["centroid"][0], 2),
            "centroid_y": round(obj["centroid"][1], 2),
            "box": obj["box"],
        })
    summary = pd.DataFrame([{
        "frame": int(frame_idx),
        "selected_classes": sorted(ACTIVE_AGENT_CLASS_NAMES),
        "agents": len(records),
        "active_agent_pixels": int(active_mask.sum()),
        "background_pixels": int(bg_mask.sum()),
        "background_ratio": float(bg_mask.mean()),
        "video_path": str(video_path) if video_path is not None else None,
    }])
    display(summary)
    display(pd.DataFrame(rows))
    return bg_mask, records, summary

### 1 - Lunar Lander


In [ ]:
active_agents_bg_1_lunar_mask, active_agents_1_lunar_records, active_agents_1_lunar_data = show_active_agent_only_segmentation_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_1_lunar.mp4",
    scene_type="lunar_lander",
)


### 2 - Car Racing


In [ ]:
active_agents_bg_2_car_mask, active_agents_2_car_records, active_agents_2_car_data = show_active_agent_only_segmentation_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_2_car.mp4",
    scene_type="car_racing",
)


### 3 - Recorded Traffic With People


In [ ]:
active_agents_bg_3_traffic_mask, active_agents_3_traffic_records, active_agents_3_traffic_data = show_active_agent_only_segmentation_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_3_traffic.mp4"
)


### 4 - Random YouTube Driving Scene


In [ ]:
active_agents_bg_4_youtube_mask, active_agents_4_youtube_records, active_agents_4_youtube_data = show_active_agent_only_segmentation_experiment(
    frames_problem4,
    problem4_frame_idx,
    "4 - Random YouTube Driving Scene - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_4_youtube.mp4"
)


### 5 - Hard Vehicle-Crowd Interaction


In [ ]:
active_agents_bg_5_hard_mask, active_agents_5_hard_records, active_agents_5_hard_data = show_active_agent_only_segmentation_experiment(
    frames_problem5,
    problem5_frame_idx,
    "5 - Hard Vehicle-Crowd Interaction - Active Agents Only: People + Vehicles",
    video_name="active_agents_only_5_hard.mp4"
)


# Background Segmentation


## Winner - Background Surface Grid Tracking (Active-Agent Inverse)

In [ ]:
# Background surface grid tracking from Active Agents Only inverse mask.
from pathlib import Path
import matplotlib.pyplot as plt
import imageio.v2 as imageio
import cv2
import numpy as np
import pandas as pd
from IPython.display import display
from PIL import Image
!pip install "imageio[ffmpeg]" -q

def ensure_surface_grid_dependencies():
    global active_agent_seg_model
    if "active_agent_seg_model" not in globals():
        from ultralytics import YOLO
        active_agent_seg_model = YOLO("yolov8n-seg.pt")
    if "ACTIVE_AGENT_CLASS_NAMES" not in globals():
        globals()["ACTIVE_AGENT_CLASS_NAMES"] = {"person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat"}
    if "mask_to_record" not in globals():
        def mask_to_record(mask, object_id):
            ys, xs = np.nonzero(mask)
            if len(xs) == 0:
                return None
            x1, x2 = int(xs.min()), int(xs.max()) + 1
            y1, y2 = int(ys.min()), int(ys.max()) + 1
            return {"object_id": object_id, "mask": mask, "box": (x1, y1, x2, y2), "area": int(mask.sum()), "centroid": (float(xs.mean()), float(ys.mean()))}
        globals()["mask_to_record"] = mask_to_record
    if "active_agent_records_from_yolo_seg" not in globals():
        def active_agent_records_from_yolo_seg(frame, model, conf_min=0.25, max_objects=40):
            result = model(frame, verbose=False, conf=conf_min)[0]
            records = []
            if result.masks is None:
                return records
            masks = result.masks.data.cpu().numpy().astype(bool)
            if masks.shape[1:] != frame.shape[:2]:
                import cv2
                masks = np.asarray([cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool) for mask in masks])
            classes = result.boxes.cls.cpu().numpy().astype(int)
            confs = result.boxes.conf.cpu().numpy()
            for mask, class_id, conf in zip(masks, classes, confs):
                class_name = result.names[int(class_id)]
                if class_name not in ACTIVE_AGENT_CLASS_NAMES or float(conf) < conf_min:
                    continue
                record = mask_to_record(mask, len(records))
                if record is None:
                    continue
                record["class_id"] = int(class_id)
                record["class_name"] = class_name
                record["confidence"] = float(conf)
                records.append(record)
            return sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
        globals()["active_agent_records_from_yolo_seg"] = active_agent_records_from_yolo_seg
    if "clip_indices" not in globals():
        def clip_indices(num_frames, clip_frames=48, seed=0):
            if num_frames <= 0:
                return np.asarray([], dtype=int)
            clip_frames = min(int(clip_frames), int(num_frames))
            rng = np.random.default_rng(seed)
            max_start = max(0, int(num_frames) - clip_frames)
            start = int(rng.integers(0, max_start + 1)) if max_start else 0
            return np.arange(start, start + clip_frames, dtype=int)
        globals()["clip_indices"] = clip_indices
    if "show_video" not in globals():
        VIDEO_DIR = Path("generated_videos")
        VIDEO_DIR.mkdir(exist_ok=True)
        def save_video(frames, path, fps=20):
            path = Path(path)
            path.parent.mkdir(parents=True, exist_ok=True)
            frames = frames.astype(np.uint8)
            try:
                imageio.mimsave(path, frames, fps=fps)
                return path
            except Exception as exc:
                fallback = path.with_suffix(".gif")
                print(f"MP4 save skipped: {type(exc).__name__}: {exc}")
                print(f"Saving GIF fallback: {fallback}")
                imageio.mimsave(fallback, frames, duration=1 / fps)
                return fallback
        def show_video(frames, name, fps=20, embed=True):
            from IPython.display import Image as DisplayImage, Video
            path = save_video(frames, VIDEO_DIR / name, fps=fps)
            if path.suffix.lower() == ".gif":
                display(DisplayImage(filename=str(path)))
            else:
                display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
            return path
        globals()["show_video"] = show_video
    if "read_video" not in globals():
        def read_video(path, max_frames=None, stride=1, start_frame=0):
            import cv2
            cap = cv2.VideoCapture(str(path))
            if start_frame:
                cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_frame))
            frames_out = []
            frame_no = 0
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                if frame_no % stride == 0:
                    frames_out.append(frame[:, :, ::-1])
                    if max_frames is not None and len(frames_out) >= max_frames:
                        break
                frame_no += 1
            cap.release()
            if not frames_out:
                raise RuntimeError(f"No frames read from {path}")
            return np.stack(frames_out)
        globals()["read_video"] = read_video
    if "read_video_sample" not in globals():
        def read_video_sample(path, max_frames=120, stride=3, start_seconds=0):
            import cv2
            cap = cv2.VideoCapture(str(path))
            fps = cap.get(cv2.CAP_PROP_FPS) or 30
            cap.release()
            return read_video(path, max_frames=max_frames, stride=stride, start_frame=int(start_seconds * fps))
        globals()["read_video_sample"] = read_video_sample
    if "frames" not in globals():
        try:
            import gymnasium as gym
            from gymnasium.envs.box2d.lunar_lander import heuristic
            env = gym.make("LunarLander-v3", render_mode="rgb_array")
            obs, _ = env.reset(seed=0)
            out = []
            for _ in range(80):
                action = int(heuristic(env.unwrapped, obs))
                obs, _, terminated, truncated, _ = env.step(action)
                out.append(env.render())
                if terminated or truncated:
                    break
            env.close()
            globals()["frames"] = np.stack(out)
            globals()["frame_idx"] = min(10, len(frames) - 1)
        except Exception as exc:
            print(f"Lunar standalone load skipped: {type(exc).__name__}: {exc}")
    if "frames_car" not in globals():
        try:
            import gymnasium as gym
            env = gym.make("CarRacing-v3", render_mode="rgb_array", continuous=False)
            obs, _ = env.reset(seed=0)
            out = []
            for _ in range(120):
                obs, _, terminated, truncated, _ = env.step(env.action_space.sample())
                out.append(env.render())
                if terminated or truncated:
                    break
            env.close()
            globals()["frames_car"] = np.stack(out)
            globals()["car_frame_idx"] = min(40, len(frames_car) - 1)
        except Exception as exc:
            print(f"Car standalone load skipped: {type(exc).__name__}: {exc}")
    if "frames_medium" not in globals() and Path("artifacts/raw_videos/traffic.avi").exists():
        globals()["frames_medium"] = read_video("artifacts/raw_videos/traffic.avi", max_frames=100, stride=1)
        globals()["traffic_frame_idx"] = min(1, len(frames_medium) - 1)
    if "frames_problem4" not in globals() and Path("artifacts/raw_videos/problem_4_youtube_random.mp4").exists():
        globals()["frames_problem4"] = read_video_sample("artifacts/raw_videos/problem_4_youtube_random.mp4", max_frames=120, stride=3, start_seconds=20)
        globals()["problem4_frame_idx"] = min(30, len(frames_problem4) - 1)
    if "frames_problem5" not in globals() and Path("artifacts/raw_videos/problem_5_youtube_hardest.mp4").exists():
        globals()["frames_problem5"] = read_video_sample("artifacts/raw_videos/problem_5_youtube_hardest.mp4", max_frames=120, stride=3, start_seconds=20)
        globals()["problem5_frame_idx"] = min(40, len(frames_problem5) - 1)


def largest_connected_component(mask, min_area=30):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    if num_labels <= 1:
        return np.zeros_like(mask, dtype=bool)
    areas = stats[1:, cv2.CC_STAT_AREA]
    best = int(np.argmax(areas)) + 1
    if areas[best - 1] < min_area:
        return np.zeros_like(mask, dtype=bool)
    return labels == best

def lunar_lander_agent_records(frame):
    rgb = frame.astype(np.int16)
    brightness = rgb.mean(axis=2)
    colorfulness = np.max(rgb, axis=2) - np.min(rgb, axis=2)
    yy = np.indices(frame.shape[:2])[0]
    mask = ((brightness > 65) | (colorfulness > 45)) & (yy < int(frame.shape[0] * 0.86))
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((5, 5), dtype=np.uint8), iterations=1).astype(bool)
    mask = largest_connected_component(mask, min_area=40)
    record = mask_to_record(mask, 0)
    if record is None:
        return []
    record.update({"class_id": -101, "class_name": "lunar_lander_agent", "confidence": 1.0, "source": "env_specific_color_geometry"})
    return [record]

def car_racing_agent_records(frame):
    rgb = frame.astype(np.int16)
    r, g, b = rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]
    h, w = frame.shape[:2]
    yy, xx = np.indices((h, w))
    red_car = (r > 120) & (r > g * 1.45) & (r > b * 1.45)
    region = (yy > int(h * 0.38)) & (yy < int(h * 0.84)) & (xx > int(w * 0.25)) & (xx < int(w * 0.75))
    mask = red_car & region
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((5, 5), dtype=np.uint8), iterations=1).astype(bool)
    mask = largest_connected_component(mask, min_area=25)
    record = mask_to_record(mask, 0)
    if record is None:
        return []
    record.update({"class_id": -102, "class_name": "car_racing_ego_car", "confidence": 1.0, "source": "env_specific_color_geometry"})
    return [record]

def active_agent_records(frame, model=None, conf_min=0.25, max_objects=40, scene_type=None):
    if scene_type == "lunar_lander":
        return lunar_lander_agent_records(frame)
    if scene_type == "car_racing":
        return car_racing_agent_records(frame)
    if model is None:
        model = active_agent_seg_model
    return active_agent_records_from_yolo_seg(frame, model=model, conf_min=conf_min, max_objects=max_objects)

def surface_grid_background_mask(frame, conf_min=0.25, dilate_px=7, scene_type=None):
    import cv2
    ensure_surface_grid_dependencies()
    records = active_agent_records(frame, model=active_agent_seg_model, conf_min=conf_min, scene_type=scene_type)
    active_mask = np.zeros(frame.shape[:2], dtype=bool)
    for record in records:
        active_mask |= record["mask"]
    if dilate_px > 0 and active_mask.any():
        kernel = np.ones((dilate_px, dilate_px), dtype=np.uint8)
        active_mask = cv2.dilate(active_mask.astype(np.uint8), kernel, iterations=1).astype(bool)
    return ~active_mask, active_mask, records

def make_surface_grid_points(mask, step=32, margin=12):
    h, w = mask.shape
    pts = []
    for y in range(margin, h - margin, step):
        for x in range(margin, w - margin, step):
            if mask[y, x]:
                pts.append((float(x), float(y)))
    return np.asarray(pts, dtype=np.float32)

def add_missing_surface_grid_points(bg_mask, points, valid, step=32, min_distance_ratio=0.72):
    candidates = make_surface_grid_points(bg_mask, step=step)
    if len(candidates) == 0:
        return points.reshape(-1, 2), valid
    points = points.reshape(-1, 2)
    valid = valid.astype(bool)
    existing = points[valid] if len(points) else np.empty((0, 2), dtype=np.float32)
    new_points = []
    min_dist_sq = float(step * min_distance_ratio) ** 2
    for candidate in candidates:
        if len(existing):
            nearest_existing = np.min(np.sum((existing - candidate) ** 2, axis=1))
            if nearest_existing < min_dist_sq:
                continue
        if new_points:
            added = np.asarray(new_points, dtype=np.float32)
            nearest_added = np.min(np.sum((added - candidate) ** 2, axis=1))
            if nearest_added < min_dist_sq:
                continue
        new_points.append(candidate)
    if not new_points:
        return points, valid
    points = np.vstack([points, np.asarray(new_points, dtype=np.float32)]) if len(points) else np.asarray(new_points, dtype=np.float32)
    valid = np.concatenate([valid, np.ones(len(new_points), dtype=bool)])
    return points, valid

def track_surface_grid_points(frames_batch, indices, step=32, scene_type=None):
    import cv2
    first = frames_batch[int(indices[0])]
    bg_mask, active_mask, records = surface_grid_background_mask(first, scene_type=scene_type)
    points0 = make_surface_grid_points(bg_mask, step=step)
    tracks = [{"frame": int(indices[0]), "points": points0, "valid": np.ones(len(points0), dtype=bool), "bg_mask": bg_mask, "active_mask": active_mask, "records": records}]
    prev_gray = cv2.cvtColor(first, cv2.COLOR_RGB2GRAY)
    prev_pts = points0.reshape(-1, 1, 2)
    valid = np.ones(len(points0), dtype=bool)
    for idx in indices[1:]:
        frame = frames_batch[int(idx)]
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        bg_mask, active_mask, records = surface_grid_background_mask(frame, scene_type=scene_type)
        if len(prev_pts) == 0:
            next_pts = prev_pts.copy()
            valid = np.zeros_like(valid)
        else:
            next_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
            if next_pts is None or status is None:
                valid = np.zeros_like(valid)
                next_pts = prev_pts.copy()
            else:
                next_xy = next_pts.reshape(-1, 2)
                h, w = bg_mask.shape
                inside = (next_xy[:, 0] >= 0) & (next_xy[:, 0] < w) & (next_xy[:, 1] >= 0) & (next_xy[:, 1] < h)
                on_background = np.zeros(len(next_xy), dtype=bool)
                rounded = np.floor(next_xy[inside]).astype(int)
                rounded[:, 0] = np.clip(rounded[:, 0], 0, w - 1)
                rounded[:, 1] = np.clip(rounded[:, 1], 0, h - 1)
                on_background[inside] = bg_mask[rounded[:, 1], rounded[:, 0]]
                valid = valid & (status.reshape(-1) == 1) & inside & on_background
        next_xy, valid = add_missing_surface_grid_points(bg_mask, next_pts.reshape(-1, 2), valid, step=step)
        next_pts = next_xy.reshape(-1, 1, 2)
        tracks.append({"frame": int(idx), "points": next_pts.reshape(-1, 2), "valid": valid.copy(), "bg_mask": bg_mask, "active_mask": active_mask, "records": records})
        prev_gray = gray
        prev_pts = next_pts
    return tracks

def render_surface_grid_frame(frame, track, prev_track=None):
    import cv2
    image = frame.copy()
    bg_mask = track["bg_mask"]
    active_mask = track["active_mask"]
    image[bg_mask] = (image[bg_mask] * 0.72 + np.array([35, 175, 105]) * 0.28).astype(np.uint8)
    image[active_mask] = (image[active_mask] * 0.35 + np.array([230, 45, 35]) * 0.65).astype(np.uint8)
    pts = track["points"]
    valid = track["valid"]
    prev_pts = prev_track["points"] if prev_track is not None and len(prev_track["points"]) == len(pts) else None
    for i, (x, y) in enumerate(pts):
        if not valid[i]:
            continue
        x_i, y_i = int(round(x)), int(round(y))
        if prev_pts is not None:
            px, py = prev_pts[i]
            cv2.arrowedLine(image, (int(round(px)), int(round(py))), (x_i, y_i), (0, 230, 255), 1, tipLength=0.25)
        cv2.circle(image, (x_i, y_i), 3, (255, 235, 0), -1)
        cv2.circle(image, (x_i, y_i), 3, (0, 0, 0), 1)
    return image

def surface_grid_summary(tracks):
    rows = []
    base = np.empty((0, 2), dtype=np.float32)
    for track in tracks:
        pts = track["points"]
        if len(pts) > len(base):
            base = np.vstack([base, pts[len(base):]]) if len(base) else pts.copy()
        rows.append(surface_grid_summary_row(track, base))
    return pd.DataFrame(rows)

def surface_grid_summary_row(track, base_points):
    pts = track["points"]
    valid = track["valid"]
    disp = pts - base_points if len(pts) == len(base_points) else np.zeros_like(pts)
    speed = np.linalg.norm(disp, axis=1) if len(disp) else np.asarray([])
    return {
        "frame": int(track["frame"]),
        "agents": int(len(track["records"])),
        "grid_points_total": int(len(pts)),
        "grid_points_tracked": int(valid.sum()),
        "background_pixels": int(track["bg_mask"].sum()),
        "median_dx_from_start": float(np.median(disp[valid, 0])) if valid.any() else np.nan,
        "median_dy_from_start": float(np.median(disp[valid, 1])) if valid.any() else np.nan,
        "median_speed_from_start": float(np.median(speed[valid])) if valid.any() else np.nan,
    }

def show_surface_grid_tracking_full_video(video_path, title, video_name, grid_step=32, max_frames=None):
    import cv2
    from IPython.display import Video
    ensure_surface_grid_dependencies()
    input_path = Path(video_path)
    output_path = Path("generated_videos") / video_name
    output_path.parent.mkdir(exist_ok=True)
    cap = cv2.VideoCapture(str(input_path))
    fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    ok, frame_bgr = cap.read()
    if not ok:
        cap.release()
        raise RuntimeError(f"No frames read from {input_path}")
    first = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    bg_mask, active_mask, records = surface_grid_background_mask(first, scene_type=scene_type)
    points0 = make_surface_grid_points(bg_mask, step=grid_step)
    track = {"frame": 0, "points": points0, "valid": np.ones(len(points0), dtype=bool), "bg_mask": bg_mask, "active_mask": active_mask, "records": records}
    base_points = points0.copy()
    rows = [surface_grid_summary_row(track, base_points)]
    rendered_preview = render_surface_grid_frame(first, track)
    writer = imageio.get_writer(output_path, fps=fps, macro_block_size=1)
    writer.append_data(rendered_preview)
    prev_gray = cv2.cvtColor(first, cv2.COLOR_RGB2GRAY)
    prev_pts = points0.reshape(-1, 1, 2)
    valid = np.ones(len(points0), dtype=bool)
    prev_track = track
    frame_no = 1
    try:
        while max_frames is None or frame_no < max_frames:
            ok, frame_bgr = cap.read()
            if not ok:
                break
            frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
            gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
            bg_mask, active_mask, records = surface_grid_background_mask(frame, scene_type=scene_type)
            if len(prev_pts) == 0:
                next_pts = prev_pts.copy()
                valid = np.zeros_like(valid)
            else:
                next_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
                if next_pts is None or status is None:
                    valid = np.zeros_like(valid)
                    next_pts = prev_pts.copy()
                else:
                    next_xy = next_pts.reshape(-1, 2)
                    h, w = bg_mask.shape
                    inside = (next_xy[:, 0] >= 0) & (next_xy[:, 0] < w) & (next_xy[:, 1] >= 0) & (next_xy[:, 1] < h)
                    on_background = np.zeros(len(next_xy), dtype=bool)
                    rounded = np.floor(next_xy[inside]).astype(int)
                    rounded[:, 0] = np.clip(rounded[:, 0], 0, w - 1)
                    rounded[:, 1] = np.clip(rounded[:, 1], 0, h - 1)
                    on_background[inside] = bg_mask[rounded[:, 1], rounded[:, 0]]
                    valid = valid & (status.reshape(-1) == 1) & inside & on_background
            next_xy, valid = add_missing_surface_grid_points(bg_mask, next_pts.reshape(-1, 2), valid, step=grid_step)
            if len(next_xy) > len(base_points):
                base_points = np.vstack([base_points, next_xy[len(base_points):]]) if len(base_points) else next_xy.copy()
            next_pts = next_xy.reshape(-1, 1, 2)
            track = {"frame": frame_no, "points": next_pts.reshape(-1, 2), "valid": valid.copy(), "bg_mask": bg_mask, "active_mask": active_mask, "records": records}
            writer.append_data(render_surface_grid_frame(frame, track, prev_track))
            rows.append(surface_grid_summary_row(track, base_points))
            prev_gray = gray
            prev_pts = next_pts
            prev_track = track
            frame_no += 1
            if frame_no % 100 == 0:
                print(f"processed {frame_no}/{total_frames or '?'} frames")
    finally:
        cap.release()
        writer.close()
    plt.figure(figsize=(8, 5))
    plt.imshow(rendered_preview)
    plt.title(title)
    plt.axis("off")
    plt.show()
    data = pd.DataFrame(rows)
    data["video_path"] = str(output_path)
    print(f"video frames: 0:{frame_no} from {input_path}")
    display(Video(str(output_path), embed=False, html_attributes="controls muted loop"))
    display(data.head(12))
    return output_path, data

def show_surface_grid_tracking_experiment(frames_batch, frame_idx, title, video_name, clip_frames=None, seed=0, grid_step=32, scene_type=None):
    ensure_surface_grid_dependencies()
    indices = np.arange(len(frames_batch), dtype=int) if clip_frames is None else clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed)
    tracks = track_surface_grid_points(frames_batch, indices, step=grid_step, scene_type=scene_type)
    selected_pos = int(np.argmin(np.abs(indices - frame_idx))) if len(indices) else 0
    rendered = [render_surface_grid_frame(frames_batch[int(i)], track, tracks[pos - 1] if pos > 0 else None) for pos, (i, track) in enumerate(zip(indices, tracks))]
    plt.figure(figsize=(8, 5))
    plt.imshow(rendered[selected_pos])
    plt.title(title)
    plt.axis("off")
    plt.show()
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    video_path = show_video(np.asarray(rendered), video_name, fps=4)
    data = surface_grid_summary(tracks)
    data["video_path"] = str(video_path)
    display(data.head(12))
    return tracks, np.asarray(rendered), data
ensure_surface_grid_dependencies()



### 1 - Lunar Lander Surface Grid Tracking


In [ ]:
bg_surface_grid_1_lunar_tracks, bg_surface_grid_1_lunar_frames, bg_surface_grid_1_lunar_data = show_surface_grid_tracking_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Background Surface Grid Tracking",
    video_name="surface_grid_1_lunar.mp4",
    scene_type="lunar_lander",
)


In [ ]:
bg_surface_grid_1_lunar_data

### 2 - Car Racing Surface Grid Tracking


In [ ]:
from pathlib import Path
import urllib.request
import cv2
import numpy as np

solved_car_path = Path("artifacts/raw_videos/2_car_racing_solved_ppo_replay.mp4")
solved_car_path.parent.mkdir(parents=True, exist_ok=True)

solved_car_url = "https://huggingface.co/Brain33/ppo-car-racing-v3/resolve/main/replay.mp4"

if not solved_car_path.exists():
    urllib.request.urlretrieve(solved_car_url, solved_car_path)

cap = cv2.VideoCapture(str(solved_car_path))
frames_car = []

while True:
    ok, frame = cap.read()
    if not ok:
        break
    frames_car.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

cap.release()

frames_car = np.stack(frames_car).astype(np.uint8)
car_frame_idx = min(120, len(frames_car) - 1)

print("Solved CarRacing replay frames:", frames_car.shape)

bg_surface_grid_2_car_tracks, bg_surface_grid_2_car_frames, bg_surface_grid_2_car_data = show_surface_grid_tracking_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing - Background Surface Grid Tracking",
    video_name="surface_grid_2_car.mp4",
    scene_type="car_racing",
)

### 3 - Recorded Traffic With People Surface Grid Tracking


In [ ]:
bg_surface_grid_3_traffic_tracks, bg_surface_grid_3_traffic_frames, bg_surface_grid_3_traffic_data = show_surface_grid_tracking_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Background Surface Grid Tracking",
    video_name="surface_grid_3_traffic.mp4",
)


### 4 - Random YouTube Driving Scene Surface Grid Tracking


In [ ]:
bg_surface_grid_4_youtube_video_path, bg_surface_grid_4_youtube_data = show_surface_grid_tracking_full_video(
    Path("artifacts/raw_videos/problem_4_youtube_random.mp4"),
    "4 - Random YouTube Driving Scene - Background Surface Grid Tracking",
    video_name="surface_grid_4_youtube.mp4",
)


### 5 - Hard Vehicle-Crowd Interaction Surface Grid Tracking


In [ ]:
bg_surface_grid_5_hard_video_path, bg_surface_grid_5_hard_data = show_surface_grid_tracking_full_video(
    Path("artifacts/raw_videos/problem_5_youtube_hardest.mp4"),
    "5 - Hard Vehicle-Crowd Interaction - Background Surface Grid Tracking",
    video_name="surface_grid_5_hard.mp4",
)


# Control Points


In [ ]:
# Control-point skeletons for active-agent blobs over inverse background.
from pathlib import Path
import cv2
import imageio.v2 as imageio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import time
from IPython.display import display

CONTROL_POINT_FRAME_SOURCES = {
    "frames": {"path": Path("generated_videos/1_lunar_lander.mp4"), "fallback": Path("generated_videos/1_lunar_lander.gif"), "idx_name": "frame_idx", "idx_value": 10, "max_frames": None, "stride": 1},
    "frames_car": {"path": Path("artifacts/raw_videos/2_car_racing_solved_ppo_replay.mp4"), "fallback": Path("generated_videos/2_car_racing_solved_full.mp4"), "idx_name": "car_frame_idx", "idx_value": 40, "max_frames": None, "stride": 1},
    "frames_medium": {"path": Path("artifacts/raw_videos/traffic.avi"), "fallback": None, "idx_name": "traffic_frame_idx", "idx_value": 1, "max_frames": None, "stride": 1},
    "frames_problem4": {"path": Path("artifacts/raw_videos/problem_4_youtube_random.mp4"), "fallback": None, "idx_name": "problem4_frame_idx", "idx_value": 30, "max_frames": 120, "stride": 3, "start_seconds": 20},
    "frames_problem5": {"path": Path("artifacts/raw_videos/problem_5_youtube_hardest.mp4"), "fallback": None, "idx_name": "problem5_frame_idx", "idx_value": 40, "max_frames": 120, "stride": 3, "start_seconds": 20},
}

def control_points_load_video_frames(path, max_frames=None, stride=1, start_seconds=0):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(path)
    if cv2 is not None:
        cap = cv2.VideoCapture(str(path))
        fps = cap.get(cv2.CAP_PROP_FPS) or 30
        if start_seconds:
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(start_seconds * fps))
        frames_out = []
        frame_no = 0
        try:
            while True:
                ok, frame = cap.read()
                if not ok:
                    break
                if frame_no % stride == 0:
                    frames_out.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
                    if max_frames is not None and len(frames_out) >= int(max_frames):
                        break
                frame_no += 1
        finally:
            cap.release()
        if frames_out:
            return np.stack(frames_out).astype(np.uint8)
    reader = imageio.get_reader(path)
    frames_out = []
    start_frame = int(start_seconds * 30)
    try:
        for frame_no, frame in enumerate(reader):
            if frame_no < start_frame:
                continue
            rel_no = frame_no - start_frame
            if rel_no % stride == 0:
                frames_out.append(np.asarray(frame)[..., :3])
                if max_frames is not None and len(frames_out) >= int(max_frames):
                    break
    finally:
        reader.close()
    if not frames_out:
        raise RuntimeError(f"No frames loaded from {path}")
    return np.stack(frames_out).astype(np.uint8)

def ensure_control_point_frames(var_name):
    spec = CONTROL_POINT_FRAME_SOURCES[var_name]
    path = spec["path"] if spec["path"].exists() else spec.get("fallback")
    if path is None or not Path(path).exists():
        raise FileNotFoundError(f"Missing frame source for {var_name}: {spec['path']}")
    if var_name not in globals():
        globals()[var_name] = control_points_load_video_frames(
            path,
            max_frames=spec.get("max_frames"),
            stride=spec.get("stride", 1),
            start_seconds=spec.get("start_seconds", 0),
        )
        print(f"loaded {var_name} from {path}:", globals()[var_name].shape)
    idx_name = spec["idx_name"]
    if idx_name not in globals():
        globals()[idx_name] = min(spec["idx_value"], len(globals()[var_name]) - 1)
        print(f"set {idx_name}:", globals()[idx_name])
    return globals()[var_name], globals()[idx_name]

CONTROL_POINTS_FAST_CLIP_FRAMES = 48
CONTROL_POINTS_FULL_VIDEO_MAX_FRAMES = 120

CONTROL_POINT_COLORS = {
    "centroid": (255, 255, 255),
    "axis_front": (0, 230, 255),
    "axis_back": (0, 150, 255),
    "axis_left": (255, 220, 0),
    "axis_right": (255, 150, 0),
    "contact_low": (255, 0, 255),
}

def ensure_control_point_dependencies():
    if "ensure_surface_grid_dependencies" in globals():
        ensure_surface_grid_dependencies()
    if "active_agent_seg_model" not in globals():
        from ultralytics import YOLO
        globals()["active_agent_seg_model"] = YOLO("yolov8n-seg.pt")
    if "ACTIVE_AGENT_CLASS_NAMES" not in globals():
        globals()["ACTIVE_AGENT_CLASS_NAMES"] = {"person", "bicycle", "car", "motorcycle", "airplane", "bus", "train", "truck", "boat"}
    if "mask_to_record" not in globals():
        def mask_to_record(mask, object_id):
            ys, xs = np.nonzero(mask)
            if len(xs) == 0:
                return None
            x1, x2 = int(xs.min()), int(xs.max()) + 1
            y1, y2 = int(ys.min()), int(ys.max()) + 1
            return {"object_id": object_id, "mask": mask, "box": (x1, y1, x2, y2), "area": int(mask.sum()), "centroid": (float(xs.mean()), float(ys.mean()))}
        globals()["mask_to_record"] = mask_to_record
    if "active_agent_records_from_yolo_seg" not in globals():
        def active_agent_records_from_yolo_seg(frame, model, conf_min=0.25, max_objects=40):
            result = model(frame, verbose=False, conf=conf_min)[0]
            records = []
            if result.masks is None:
                return records
            masks = result.masks.data.cpu().numpy().astype(bool)
            if masks.shape[1:] != frame.shape[:2]:
                masks = np.asarray([cv2.resize(mask.astype(np.uint8), (frame.shape[1], frame.shape[0]), interpolation=cv2.INTER_NEAREST).astype(bool) for mask in masks])
            classes = result.boxes.cls.cpu().numpy().astype(int)
            confs = result.boxes.conf.cpu().numpy()
            for mask, class_id, conf in zip(masks, classes, confs):
                class_name = result.names[int(class_id)]
                if class_name not in ACTIVE_AGENT_CLASS_NAMES or float(conf) < conf_min:
                    continue
                record = mask_to_record(mask, len(records))
                if record is None:
                    continue
                record["class_id"] = int(class_id)
                record["class_name"] = class_name
                record["confidence"] = float(conf)
                records.append(record)
            return sorted(records, key=lambda obj: obj["area"], reverse=True)[:max_objects]
        globals()["active_agent_records_from_yolo_seg"] = active_agent_records_from_yolo_seg
    if "show_video" not in globals():
        VIDEO_DIR = Path("generated_videos")
        VIDEO_DIR.mkdir(exist_ok=True)
        def save_video(frames, path, fps=20):
            path = Path(path)
            path.parent.mkdir(parents=True, exist_ok=True)
            imageio.mimsave(path, frames.astype(np.uint8), fps=fps)
            return path
        def show_video(frames, name, fps=20, embed=True):
            from IPython.display import Video
            path = save_video(frames, VIDEO_DIR / name, fps=fps)
            display(Video(str(path), embed=embed, html_attributes="controls muted loop"))
            return path
        globals()["show_video"] = show_video
    if "clip_indices" not in globals():
        def clip_indices(num_frames, clip_frames=48, seed=0):
            clip_frames = min(int(clip_frames), int(num_frames))
            return np.arange(clip_frames, dtype=int)
        globals()["clip_indices"] = clip_indices


def control_points_largest_connected_component(mask, min_area=30):
    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    if num_labels <= 1:
        return np.zeros_like(mask, dtype=bool)
    areas = stats[1:, cv2.CC_STAT_AREA]
    best = int(np.argmax(areas)) + 1
    if areas[best - 1] < min_area:
        return np.zeros_like(mask, dtype=bool)
    return labels == best

def lunar_lander_agent_records(frame):
    rgb = frame.astype(np.int16)
    h, w = frame.shape[:2]
    yy, xx = np.indices((h, w))
    brightness = rgb.mean(axis=2)
    colorfulness = np.max(rgb, axis=2) - np.min(rgb, axis=2)
    # The lander is a compact bright/colored body in the playable sky area. Avoid the top UI/ceiling
    # and the lower terrain line, then choose a plausible compact component near the screen center.
    candidate = ((brightness > 85) | (colorfulness > 55))
    playable = (yy > int(h * 0.08)) & (yy < int(h * 0.78)) & (xx > int(w * 0.12)) & (xx < int(w * 0.88))
    mask = candidate & playable
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((3, 3), dtype=np.uint8), iterations=1)
    num_labels, labels, stats, centroids = cv2.connectedComponentsWithStats(mask.astype(np.uint8), connectivity=8)
    if num_labels <= 1:
        return []
    screen_center = np.array([w * 0.5, h * 0.42], dtype=np.float32)
    best_label = None
    best_score = np.inf
    for label in range(1, num_labels):
        area = int(stats[label, cv2.CC_STAT_AREA])
        x, y = int(stats[label, cv2.CC_STAT_LEFT]), int(stats[label, cv2.CC_STAT_TOP])
        bw, bh = int(stats[label, cv2.CC_STAT_WIDTH]), int(stats[label, cv2.CC_STAT_HEIGHT])
        if area < 25 or area > 3500:
            continue
        if bw > int(w * 0.28) or bh > int(h * 0.28):
            continue
        centroid = centroids[label].astype(np.float32)
        center_dist = np.linalg.norm((centroid - screen_center) / np.array([w, h], dtype=np.float32))
        compact_penalty = 0.0008 * area + 0.002 * max(bw, bh)
        score = center_dist + compact_penalty
        if score < best_score:
            best_score = score
            best_label = label
    if best_label is None:
        mask = control_points_largest_connected_component(mask, min_area=25)
    else:
        mask = labels == best_label
    record = mask_to_record(mask, 0)
    if record is None:
        return []
    record.update({"class_id": -101, "class_name": "lunar_lander_agent", "confidence": 1.0, "source": "env_specific_color_geometry"})
    return [record]

def car_racing_agent_records(frame):
    rgb = frame.astype(np.int16)
    r, g, b = rgb[:, :, 0], rgb[:, :, 1], rgb[:, :, 2]
    h, w = frame.shape[:2]
    yy, xx = np.indices((h, w))
    red_car = (r > 120) & (r > g * 1.45) & (r > b * 1.45)
    region = (yy > int(h * 0.38)) & (yy < int(h * 0.84)) & (xx > int(w * 0.25)) & (xx < int(w * 0.75))
    mask = red_car & region
    mask = cv2.morphologyEx(mask.astype(np.uint8), cv2.MORPH_CLOSE, np.ones((5, 5), dtype=np.uint8), iterations=1).astype(bool)
    mask = control_points_largest_connected_component(mask, min_area=25)
    record = mask_to_record(mask, 0)
    if record is None:
        return []
    record.update({"class_id": -102, "class_name": "car_racing_ego_car", "confidence": 1.0, "source": "env_specific_color_geometry"})
    return [record]

def active_agent_records(frame, model=None, conf_min=0.25, max_objects=40, scene_type=None):
    if scene_type == "lunar_lander":
        return lunar_lander_agent_records(frame)
    if scene_type == "car_racing":
        return car_racing_agent_records(frame)
    if model is None:
        model = active_agent_seg_model
    return active_agent_records_from_yolo_seg(frame, model=model, conf_min=conf_min, max_objects=max_objects)

def mask_control_points(mask):
    mask = mask.astype(bool)
    ys, xs = np.nonzero(mask)
    if len(xs) == 0:
        return {}
    coords = np.column_stack([xs.astype(np.float32), ys.astype(np.float32)])
    centroid = coords.mean(axis=0)
    centered = coords - centroid
    if len(coords) >= 3 and np.any(centered):
        cov = np.cov(centered, rowvar=False)
        eigvals, eigvecs = np.linalg.eigh(cov)
        major = eigvecs[:, int(np.argmax(eigvals))]
        minor = np.array([-major[1], major[0]], dtype=np.float32)
    else:
        major = np.array([1.0, 0.0], dtype=np.float32)
        minor = np.array([0.0, 1.0], dtype=np.float32)
    major_proj = centered @ major
    minor_proj = centered @ minor
    bottom_threshold = np.percentile(coords[:, 1], 95)
    bottom = coords[coords[:, 1] >= bottom_threshold]
    contact_low = bottom[np.argmin(np.abs(bottom[:, 0] - np.median(bottom[:, 0])))] if len(bottom) else coords[np.argmax(coords[:, 1])]
    return {
        "centroid": tuple(centroid),
        "axis_front": tuple(coords[int(np.argmax(major_proj))]),
        "axis_back": tuple(coords[int(np.argmin(major_proj))]),
        "axis_left": tuple(coords[int(np.argmin(minor_proj))]),
        "axis_right": tuple(coords[int(np.argmax(minor_proj))]),
        "contact_low": tuple(contact_low),
    }

def add_control_points_to_records(records):
    for record in records:
        record["control_points"] = mask_control_points(record["mask"])
    return records

def active_agent_records_with_control_points(frame, conf_min=0.25, scene_type=None):
    ensure_control_point_dependencies()
    records = active_agent_records(frame, model=active_agent_seg_model, conf_min=conf_min, scene_type=scene_type)
    return add_control_points_to_records(records)

def render_control_points_overlay(image, records):
    image = image.copy()
    for record in records:
        mask = record["mask"].astype(np.uint8)
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(image, contours, -1, (255, 255, 255), 1)
        points = record.get("control_points", {})
        centroid = points.get("centroid")
        if centroid is not None:
            c = (int(round(centroid[0])), int(round(centroid[1])))
            for a, b in [("axis_front", "axis_back"), ("axis_left", "axis_right")]:
                if a in points and b in points:
                    p1 = (int(round(points[a][0])), int(round(points[a][1])))
                    p2 = (int(round(points[b][0])), int(round(points[b][1])))
                    cv2.line(image, p1, p2, (255, 255, 255), 1)
            if "contact_low" in points:
                p = (int(round(points["contact_low"][0])), int(round(points["contact_low"][1])))
                cv2.line(image, c, p, (255, 255, 255), 1)
        for name, point in points.items():
            x, y = int(round(point[0])), int(round(point[1]))
            color = CONTROL_POINT_COLORS.get(name, (255, 255, 255))
            cv2.circle(image, (x, y), 4, color, -1)
            cv2.circle(image, (x, y), 4, (0, 0, 0), 1)
        x1, y1, x2, y2 = record.get("box", (0, 0, 0, 0))
        label = f"T{record.get('track_id', -1)} {record.get('class_name', 'agent')} {record.get('confidence', 0):.2f}"
        cv2.putText(image, label, (int(x1), max(15, int(y1) - 5)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, (255, 255, 255), 1, cv2.LINE_AA)
    return image

def render_control_points_frame(frame, records):
    active_mask = np.zeros(frame.shape[:2], dtype=bool)
    for record in records:
        active_mask |= record["mask"]
    bg_mask = ~active_mask
    base = frame.copy()
    base[bg_mask] = (base[bg_mask] * 0.72 + np.array([35, 175, 105]) * 0.28).astype(np.uint8)
    base[active_mask] = (base[active_mask] * 0.35 + np.array([230, 45, 35]) * 0.65).astype(np.uint8)
    return render_control_points_overlay(base, records)


def control_points_box_iou(box_a, box_b):
    ax1, ay1, ax2, ay2 = box_a
    bx1, by1, bx2, by2 = box_b
    ix1, iy1 = max(ax1, bx1), max(ay1, by1)
    ix2, iy2 = min(ax2, bx2), min(ay2, by2)
    iw, ih = max(0.0, ix2 - ix1), max(0.0, iy2 - iy1)
    inter = iw * ih
    area_a = max(0.0, ax2 - ax1) * max(0.0, ay2 - ay1)
    area_b = max(0.0, bx2 - bx1) * max(0.0, by2 - by1)
    denom = area_a + area_b - inter
    return float(inter / denom) if denom > 0 else 0.0

def assign_control_point_track_ids(records_by_frame, iou_threshold=0.12, centroid_distance_threshold=80.0, max_missing=8):
    active_tracks = []
    next_track_id = 0
    for frame_pos, records in enumerate(records_by_frame):
        candidates = []
        for track_idx, track in enumerate(active_tracks):
            if track["missing"] > max_missing:
                continue
            for record_idx, record in enumerate(records):
                if record.get("class_name") != track.get("class_name"):
                    continue
                iou = control_points_box_iou(track["box"], record.get("box", (0, 0, 0, 0)))
                cx, cy = record.get("centroid", (np.nan, np.nan))
                tx, ty = track.get("centroid", (np.nan, np.nan))
                centroid_distance = float(np.hypot(cx - tx, cy - ty)) if np.isfinite(cx) and np.isfinite(tx) else np.inf
                if iou >= iou_threshold or centroid_distance <= centroid_distance_threshold:
                    score = iou - 0.0025 * centroid_distance - 0.03 * track["missing"]
                    candidates.append((score, track_idx, record_idx, iou, centroid_distance))
        used_tracks = set()
        used_records = set()
        for score, track_idx, record_idx, iou, centroid_distance in sorted(candidates, reverse=True):
            if track_idx in used_tracks or record_idx in used_records:
                continue
            track = active_tracks[track_idx]
            record = records[record_idx]
            record["track_id"] = int(track["track_id"])
            record["track_iou"] = float(iou)
            record["track_centroid_distance"] = float(centroid_distance)
            record["track_age"] = int(track["age"] + 1)
            track.update({
                "box": record.get("box", track["box"]),
                "centroid": record.get("centroid", track["centroid"]),
                "class_name": record.get("class_name"),
                "missing": 0,
                "age": track["age"] + 1,
            })
            used_tracks.add(track_idx)
            used_records.add(record_idx)
        for record_idx, record in enumerate(records):
            if record_idx in used_records:
                continue
            record["track_id"] = int(next_track_id)
            record["track_iou"] = np.nan
            record["track_centroid_distance"] = np.nan
            record["track_age"] = 1
            active_tracks.append({
                "track_id": int(next_track_id),
                "class_name": record.get("class_name"),
                "box": record.get("box", (0, 0, 0, 0)),
                "centroid": record.get("centroid", (np.nan, np.nan)),
                "missing": 0,
                "age": 1,
            })
            next_track_id += 1
        for track_idx, track in enumerate(active_tracks):
            if track_idx not in used_tracks:
                track["missing"] += 1
        active_tracks = [track for track in active_tracks if track["missing"] <= max_missing]
    return records_by_frame

def control_points_dataframe(frame_no, records):
    rows = []
    for record in records:
        x1, y1, x2, y2 = record.get("box", (np.nan, np.nan, np.nan, np.nan))
        for point_name, point in record.get("control_points", {}).items():
            rows.append({
                "frame": int(frame_no),
                "object_id": int(record.get("object_id", -1)),
                "track_id": int(record.get("track_id", -1)),
                "track_age": int(record.get("track_age", 1)),
                "track_iou": record.get("track_iou", np.nan),
                "track_centroid_distance": record.get("track_centroid_distance", np.nan),
                "class_name": record.get("class_name"),
                "confidence": float(record.get("confidence", np.nan)),
                "box_x1": float(x1),
                "box_y1": float(y1),
                "box_x2": float(x2),
                "box_y2": float(y2),
                "area": int(record.get("area", 0)),
                "point_name": point_name,
                "x": float(point[0]),
                "y": float(point[1]),
                "method": "mask PCA skeleton + low contact point",
            })
    return rows


def control_points_background_dataframe(track, prev_track=None):
    rows = []
    points = np.asarray(track.get("points", []), dtype=np.float32).reshape(-1, 2)
    valid = np.asarray(track.get("valid", np.zeros(len(points), dtype=bool)), dtype=bool)
    prev_points = None
    prev_valid = None
    if prev_track is not None:
        prev_points = np.asarray(prev_track.get("points", []), dtype=np.float32).reshape(-1, 2)
        prev_valid = np.asarray(prev_track.get("valid", np.zeros(len(prev_points), dtype=bool)), dtype=bool)
    for point_id, (x, y) in enumerate(points):
        has_prev = prev_points is not None and point_id < len(prev_points)
        px, py = prev_points[point_id] if has_prev else (x, y)
        prev_is_valid = bool(prev_valid[point_id]) if has_prev and point_id < len(prev_valid) else False
        is_valid = bool(valid[point_id]) if point_id < len(valid) else False
        dx = float(x - px) if has_prev else 0.0
        dy = float(y - py) if has_prev else 0.0
        rows.append({
            "frame": int(track.get("frame", -1)),
            "background_point_id": int(point_id),
            "x": float(x),
            "y": float(y),
            "valid": is_valid,
            "has_prev": bool(has_prev),
            "prev_x": float(px),
            "prev_y": float(py),
            "prev_valid": prev_is_valid,
            "dx": dx,
            "dy": dy,
            "speed": float(np.hypot(dx, dy)),
            "method": "surface grid + Lucas-Kanade optical flow",
        })
    return rows


def control_points_background_mask(frame, conf_min=0.25, scene_type=None, dilate_px=7):
    records = active_agent_records(frame, model=active_agent_seg_model, conf_min=conf_min, scene_type=scene_type)
    active_mask = np.zeros(frame.shape[:2], dtype=bool)
    for record in records:
        active_mask |= record["mask"]
    if dilate_px > 0 and active_mask.any():
        kernel = np.ones((dilate_px, dilate_px), dtype=np.uint8)
        active_mask = cv2.dilate(active_mask.astype(np.uint8), kernel, iterations=1).astype(bool)
    return ~active_mask, active_mask, records

def control_points_make_surface_grid_points(mask, step=32, margin=12):
    h, w = mask.shape
    pts = []
    for y in range(margin, h - margin, step):
        for x in range(margin, w - margin, step):
            if mask[y, x]:
                pts.append((float(x), float(y)))
    return np.asarray(pts, dtype=np.float32)

def control_points_add_missing_surface_grid_points(bg_mask, points, valid, step=32, min_distance_ratio=0.72):
    candidates = control_points_make_surface_grid_points(bg_mask, step=step)
    if len(candidates) == 0:
        return points.reshape(-1, 2), valid
    points = points.reshape(-1, 2)
    valid = valid.astype(bool)
    existing = points[valid] if len(points) else np.empty((0, 2), dtype=np.float32)
    new_points = []
    min_dist_sq = float(step * min_distance_ratio) ** 2
    for candidate in candidates:
        if len(existing) and np.min(np.sum((existing - candidate) ** 2, axis=1)) < min_dist_sq:
            continue
        if new_points:
            added = np.asarray(new_points, dtype=np.float32)
            if np.min(np.sum((added - candidate) ** 2, axis=1)) < min_dist_sq:
                continue
        new_points.append(candidate)
    if not new_points:
        return points, valid
    points = np.vstack([points, np.asarray(new_points, dtype=np.float32)]) if len(points) else np.asarray(new_points, dtype=np.float32)
    valid = np.concatenate([valid, np.ones(len(new_points), dtype=bool)])
    return points, valid

def control_points_track_surface_grid_points(frames_batch, indices, step=32, scene_type=None):
    first = frames_batch[int(indices[0])]
    bg_mask, active_mask, records = control_points_background_mask(first, scene_type=scene_type)
    points0 = control_points_make_surface_grid_points(bg_mask, step=step)
    tracks = [{"frame": int(indices[0]), "points": points0, "valid": np.ones(len(points0), dtype=bool), "bg_mask": bg_mask, "active_mask": active_mask, "records": records}]
    prev_gray = cv2.cvtColor(first, cv2.COLOR_RGB2GRAY)
    prev_pts = points0.reshape(-1, 1, 2)
    valid = np.ones(len(points0), dtype=bool)
    for pos, idx in enumerate(indices[1:], start=1):
        if pos == 1 or pos % 25 == 0 or pos == len(indices) - 1:
            print(f"tracking control-point background {pos + 1}/{len(indices)} frames")
        frame = frames_batch[int(idx)]
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        bg_mask, active_mask, records = control_points_background_mask(frame, scene_type=scene_type)
        if len(prev_pts) == 0:
            next_pts = prev_pts.copy()
            valid = np.zeros_like(valid)
        else:
            next_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
            if next_pts is None or status is None:
                valid = np.zeros_like(valid)
                next_pts = prev_pts.copy()
            else:
                next_xy = next_pts.reshape(-1, 2)
                h, w = bg_mask.shape
                inside = (next_xy[:, 0] >= 0) & (next_xy[:, 0] < w) & (next_xy[:, 1] >= 0) & (next_xy[:, 1] < h)
                on_background = np.zeros(len(next_xy), dtype=bool)
                rounded = np.floor(next_xy[inside]).astype(int)
                rounded[:, 0] = np.clip(rounded[:, 0], 0, w - 1)
                rounded[:, 1] = np.clip(rounded[:, 1], 0, h - 1)
                on_background[inside] = bg_mask[rounded[:, 1], rounded[:, 0]]
                valid = valid & (status.reshape(-1) == 1) & inside & on_background
        next_xy, valid = control_points_add_missing_surface_grid_points(bg_mask, next_pts.reshape(-1, 2), valid, step=step)
        next_pts = next_xy.reshape(-1, 1, 2)
        tracks.append({"frame": int(idx), "points": next_pts.reshape(-1, 2), "valid": valid.copy(), "bg_mask": bg_mask, "active_mask": active_mask, "records": records})
        prev_gray = gray
        prev_pts = next_pts
    return tracks

def control_points_render_surface_grid_frame(frame, track, prev_track=None):
    image = frame.copy()
    bg_mask = track["bg_mask"]
    active_mask = track["active_mask"]
    image[bg_mask] = (image[bg_mask] * 0.72 + np.array([35, 175, 105]) * 0.28).astype(np.uint8)
    image[active_mask] = (image[active_mask] * 0.35 + np.array([230, 45, 35]) * 0.65).astype(np.uint8)
    pts = track["points"]
    valid = track["valid"]
    prev_pts = prev_track["points"] if prev_track is not None and len(prev_track["points"]) == len(pts) else None
    for i, (x, y) in enumerate(pts):
        if not valid[i]:
            continue
        x_i, y_i = int(round(x)), int(round(y))
        if prev_pts is not None:
            px, py = prev_pts[i]
            cv2.arrowedLine(image, (int(round(px)), int(round(py))), (x_i, y_i), (0, 230, 255), 1, tipLength=0.25)
        cv2.circle(image, (x_i, y_i), 3, (255, 235, 0), -1)
        cv2.circle(image, (x_i, y_i), 3, (0, 0, 0), 1)
    return image

def show_control_points_with_background_experiment(frames_batch, frame_idx, title, video_name, clip_frames=None, seed=0, fps=6, grid_step=32, scene_type=None):
    ensure_control_point_dependencies()
    indices = np.arange(len(frames_batch), dtype=int) if clip_frames is None else clip_indices(len(frames_batch), clip_frames=clip_frames, seed=seed)
    tracks = control_points_track_surface_grid_points(frames_batch, indices, step=grid_step, scene_type=scene_type)
    records_by_frame = [add_control_points_to_records(track["records"]) for track in tracks]
    records_by_frame = assign_control_point_track_ids(records_by_frame)
    rendered = []
    rows = []
    background_rows = []
    selected_pos = int(np.argmin(np.abs(indices - frame_idx))) if len(indices) else 0
    for pos, idx in enumerate(indices):
        if pos == 0 or (pos + 1) % 25 == 0 or pos == len(indices) - 1:
            print(f"rendering control points {pos + 1}/{len(indices)} frames")
        frame = frames_batch[int(idx)]
        track = tracks[pos]
        records = records_by_frame[pos]
        base = control_points_render_surface_grid_frame(frame, track, tracks[pos - 1] if pos > 0 else None)
        rendered.append(render_control_points_overlay(base, records))
        rows.extend(control_points_dataframe(int(idx), records))
        background_rows.extend(control_points_background_dataframe(track, tracks[pos - 1] if pos > 0 else None))
    rendered = np.asarray(rendered, dtype=np.uint8)
    plt.figure(figsize=(8, 5))
    plt.imshow(rendered[selected_pos])
    plt.title(title)
    plt.axis("off")
    plt.show()
    print(f"video frames: {int(indices[0]) if len(indices) else 0}:{int(indices[-1]) + 1 if len(indices) else 0}")
    video_path = show_video(rendered, video_name, fps=fps)
    data = pd.DataFrame(rows)
    data["video_path"] = str(video_path)
    background_data = pd.DataFrame(background_rows)
    background_data["video_path"] = str(video_path)
    display(data.head(30))
    display(background_data[background_data["frame"] > int(indices[0])].head(30))
    return rendered, data, background_data


def format_seconds(seconds):
    if seconds is None or not np.isfinite(seconds):
        return "?"
    seconds = int(max(0, seconds))
    hours, rem = divmod(seconds, 3600)
    minutes, secs = divmod(rem, 60)
    if hours:
        return f"{hours}h {minutes:02d}m {secs:02d}s"
    return f"{minutes}m {secs:02d}s"

def show_control_points_full_video_experiment(video_path, title, video_name, fps=None, grid_step=32, scene_type=None, max_frames=None):
    ensure_control_point_dependencies()
    input_path = Path(video_path)
    output_path = Path("generated_videos") / video_name
    output_path.parent.mkdir(parents=True, exist_ok=True)
    cap = cv2.VideoCapture(str(input_path))
    source_fps = cap.get(cv2.CAP_PROP_FPS) or 30
    total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    effective_total_frames = min(total_frames, int(max_frames)) if max_frames is not None and total_frames else total_frames
    out_fps = fps or source_fps
    start_time = time.time()
    ok, frame_bgr = cap.read()
    if not ok:
        cap.release()
        raise RuntimeError(f"No frames read from {input_path}")
    first = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    bg_mask, active_mask, records = control_points_background_mask(first, scene_type=scene_type)
    points0 = control_points_make_surface_grid_points(bg_mask, step=grid_step)
    track = {"frame": 0, "points": points0, "valid": np.ones(len(points0), dtype=bool), "bg_mask": bg_mask, "active_mask": active_mask, "records": records}
    prev_gray = cv2.cvtColor(first, cv2.COLOR_RGB2GRAY)
    prev_pts = points0.reshape(-1, 1, 2)
    valid = np.ones(len(points0), dtype=bool)
    prev_track = track
    active_tracks = []
    next_track_id = 0
    rows = []
    background_rows = []
    writer = imageio.get_writer(output_path, fps=out_fps, macro_block_size=1)
    preview = None
    frame_no = 0
    try:
        while True:
            if frame_no > 0:
                if max_frames is not None and frame_no >= max_frames:
                    break
                ok, frame_bgr = cap.read()
                if not ok:
                    break
                frame = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
                gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
                bg_mask, active_mask, records = control_points_background_mask(frame, scene_type=scene_type)
                if len(prev_pts) == 0:
                    next_pts = prev_pts.copy()
                    valid = np.zeros_like(valid)
                else:
                    next_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_gray, gray, prev_pts, None, winSize=(21, 21), maxLevel=3, criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
                    if next_pts is None or status is None:
                        valid = np.zeros_like(valid)
                        next_pts = prev_pts.copy()
                    else:
                        next_xy = next_pts.reshape(-1, 2)
                        h, w = bg_mask.shape
                        inside = (next_xy[:, 0] >= 0) & (next_xy[:, 0] < w) & (next_xy[:, 1] >= 0) & (next_xy[:, 1] < h)
                        on_background = np.zeros(len(next_xy), dtype=bool)
                        rounded = np.floor(next_xy[inside]).astype(int)
                        rounded[:, 0] = np.clip(rounded[:, 0], 0, w - 1)
                        rounded[:, 1] = np.clip(rounded[:, 1], 0, h - 1)
                        on_background[inside] = bg_mask[rounded[:, 1], rounded[:, 0]]
                        valid = valid & (status.reshape(-1) == 1) & inside & on_background
                next_xy, valid = control_points_add_missing_surface_grid_points(bg_mask, next_pts.reshape(-1, 2), valid, step=grid_step)
                next_pts = next_xy.reshape(-1, 1, 2)
                track = {"frame": frame_no, "points": next_pts.reshape(-1, 2), "valid": valid.copy(), "bg_mask": bg_mask, "active_mask": active_mask, "records": records}
            else:
                frame = first
                gray = prev_gray
                next_pts = prev_pts

            records = add_control_points_to_records(track["records"])
            records_by_frame = assign_control_point_track_ids([records]) if frame_no == 0 else [records]
            if frame_no == 0:
                # Initialize persistent tracker state from the first-frame assignment.
                active_tracks = []
                for record in records_by_frame[0]:
                    active_tracks.append({"track_id": int(record["track_id"]), "class_name": record.get("class_name"), "box": record.get("box"), "centroid": record.get("centroid"), "missing": 0, "age": int(record.get("track_age", 1))})
                next_track_id = max([t["track_id"] for t in active_tracks], default=-1) + 1
                records = records_by_frame[0]
            else:
                # Inline persistent assignment for streaming, using the previous active tracks.
                candidates = []
                for track_idx, live_track in enumerate(active_tracks):
                    if live_track["missing"] > 8:
                        continue
                    for record_idx, record in enumerate(records):
                        if record.get("class_name") != live_track.get("class_name"):
                            continue
                        iou = control_points_box_iou(live_track["box"], record.get("box", (0, 0, 0, 0)))
                        cx, cy = record.get("centroid", (np.nan, np.nan))
                        tx, ty = live_track.get("centroid", (np.nan, np.nan))
                        centroid_distance = float(np.hypot(cx - tx, cy - ty)) if np.isfinite(cx) and np.isfinite(tx) else np.inf
                        if iou >= 0.12 or centroid_distance <= 80.0:
                            score = iou - 0.0025 * centroid_distance - 0.03 * live_track["missing"]
                            candidates.append((score, track_idx, record_idx, iou, centroid_distance))
                used_tracks, used_records = set(), set()
                for _, track_idx, record_idx, iou, centroid_distance in sorted(candidates, reverse=True):
                    if track_idx in used_tracks or record_idx in used_records:
                        continue
                    live_track = active_tracks[track_idx]
                    record = records[record_idx]
                    record["track_id"] = int(live_track["track_id"])
                    record["track_iou"] = float(iou)
                    record["track_centroid_distance"] = float(centroid_distance)
                    record["track_age"] = int(live_track["age"] + 1)
                    live_track.update({"box": record.get("box"), "centroid": record.get("centroid"), "class_name": record.get("class_name"), "missing": 0, "age": live_track["age"] + 1})
                    used_tracks.add(track_idx)
                    used_records.add(record_idx)
                for record_idx, record in enumerate(records):
                    if record_idx in used_records:
                        continue
                    record["track_id"] = int(next_track_id)
                    record["track_iou"] = np.nan
                    record["track_centroid_distance"] = np.nan
                    record["track_age"] = 1
                    active_tracks.append({"track_id": int(next_track_id), "class_name": record.get("class_name"), "box": record.get("box"), "centroid": record.get("centroid"), "missing": 0, "age": 1})
                    next_track_id += 1
                for track_idx, live_track in enumerate(active_tracks):
                    if track_idx not in used_tracks:
                        live_track["missing"] += 1
                active_tracks = [live_track for live_track in active_tracks if live_track["missing"] <= 8]

            base = control_points_render_surface_grid_frame(frame, track, prev_track if frame_no > 0 else None)
            rendered_frame = render_control_points_overlay(base, records)
            if preview is None:
                preview = rendered_frame.copy()
            writer.append_data(rendered_frame)
            rows.extend(control_points_dataframe(frame_no, records))
            background_rows.extend(control_points_background_dataframe(track, prev_track if frame_no > 0 else None))
            if frame_no == 0 or (frame_no + 1) % 25 == 0 or (effective_total_frames and frame_no + 1 == effective_total_frames):
                done = frame_no + 1
                total_label = effective_total_frames or "?"
                elapsed = time.time() - start_time
                fps_done = done / elapsed if elapsed > 0 else 0.0
                percent = (100.0 * done / effective_total_frames) if effective_total_frames else np.nan
                eta = ((effective_total_frames - done) / fps_done) if effective_total_frames and fps_done > 0 else np.nan
                percent_label = f"{percent:.1f}%" if np.isfinite(percent) else "?%"
                print(f"control points full video: {done}/{total_label} ({percent_label}) | elapsed {format_seconds(elapsed)} | ETA {format_seconds(eta)} | {fps_done:.2f} fps")
            prev_gray = gray
            prev_pts = next_pts
            prev_track = track
            frame_no += 1
    finally:
        cap.release()
        writer.close()
    data = pd.DataFrame(rows)
    data["video_path"] = str(output_path)
    background_data = pd.DataFrame(background_rows)
    background_data["video_path"] = str(output_path)
    plt.figure(figsize=(8, 5))
    plt.imshow(preview)
    plt.title(title)
    plt.axis("off")
    plt.show()
    print(f"video frames: 0:{frame_no} from {input_path}")
    display(data.head(30))
    display(background_data[background_data["frame"] > 0].head(30))
    return output_path, data, background_data

ensure_control_point_dependencies()


### 1 - Lunar Lander Control Points


In [ ]:
frames, frame_idx = ensure_control_point_frames("frames")
control_points_1_lunar_frames, control_points_1_lunar_data, control_points_1_lunar_background_data = show_control_points_with_background_experiment(
    frames,
    frame_idx,
    "1 - Lunar Lander - Control Points",
    video_name="control_points_1_lunar.mp4",
    scene_type="lunar_lander",
    clip_frames=CONTROL_POINTS_FAST_CLIP_FRAMES,
)


### 2 - Car Racing Control Points


In [ ]:
frames_car, car_frame_idx = ensure_control_point_frames("frames_car")
control_points_2_car_frames, control_points_2_car_data, control_points_2_car_background_data = show_control_points_with_background_experiment(
    frames_car,
    car_frame_idx,
    "2 - Car Racing Solved Replay - Control Points",
    video_name="control_points_2_car_solved_full.mp4",
    scene_type="car_racing",
    clip_frames=CONTROL_POINTS_FAST_CLIP_FRAMES,
)


### 3 - Recorded Traffic With People Control Points


In [ ]:
frames_medium, traffic_frame_idx = ensure_control_point_frames("frames_medium")
control_points_3_traffic_frames, control_points_3_traffic_data, control_points_3_traffic_background_data = show_control_points_with_background_experiment(
    frames_medium,
    traffic_frame_idx,
    "3 - Recorded Traffic With People - Control Points",
    video_name="control_points_3_traffic.mp4",
    clip_frames=CONTROL_POINTS_FAST_CLIP_FRAMES,
)


### 4 - Random YouTube Driving Scene Control Points


In [ ]:
control_points_4_youtube_video_path, control_points_4_youtube_data, control_points_4_youtube_background_data = show_control_points_full_video_experiment(
    Path("artifacts/raw_videos/problem_4_youtube_random.mp4"),
    "4 - Random YouTube Driving Scene - Control Points",
    video_name="control_points_4_youtube.mp4",
    max_frames=CONTROL_POINTS_FULL_VIDEO_MAX_FRAMES,
)


### 5 - Hard Vehicle-Crowd Interaction Control Points


In [ ]:
control_points_5_hard_video_path, control_points_5_hard_data, control_points_5_hard_background_data = show_control_points_full_video_experiment(
    Path("artifacts/raw_videos/problem_5_youtube_hardest.mp4"),
    "5 - Hard Vehicle-Crowd Interaction - Control Points",
    video_name="control_points_5_hard.mp4",
    max_frames=CONTROL_POINTS_FULL_VIDEO_MAX_FRAMES,
)


## Save Control Points Artifacts


In [ ]:
from pathlib import Path
import shutil
import imageio.v2 as imageio

CONTROL_POINTS_ARTIFACT_DIR = Path("artifacts/research/control_points")
BACKGROUND_POINTS_ARTIFACT_DIR = Path("artifacts/research/background_points")
CONTROL_POINTS_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
BACKGROUND_POINTS_ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)



def save_control_points_background_artifact(data, artifact_name):
    table_path = BACKGROUND_POINTS_ARTIFACT_DIR / f"{artifact_name}__background_points.csv"
    data.to_csv(table_path, index=False)
    print("saved background table:", table_path)
    return table_path

def save_control_points_artifact(video_or_frames, data, artifact_name, fps=6):
    video_path = CONTROL_POINTS_ARTIFACT_DIR / f"{artifact_name}.mp4"
    table_path = CONTROL_POINTS_ARTIFACT_DIR / f"{artifact_name}.csv"
    if isinstance(video_or_frames, (str, Path)):
        shutil.copy2(Path(video_or_frames), video_path)
    else:
        imageio.mimsave(video_path, video_or_frames.astype(np.uint8), fps=fps)
    data.to_csv(table_path, index=False)
    print("saved video:", video_path)
    print("saved table:", table_path)
    return video_path, table_path

control_points_artifacts = {
    "1_lunar_lander": save_control_points_artifact(
        control_points_1_lunar_frames,
        control_points_1_lunar_data,
        "1__control_points__lunar_lander",
    ),
    "2_car_racing": save_control_points_artifact(
        control_points_2_car_frames,
        control_points_2_car_data,
        "2__control_points__car_racing_solved_full",
    ),
    "3_recorded_traffic_people": save_control_points_artifact(
        control_points_3_traffic_frames,
        control_points_3_traffic_data,
        "3__control_points__recorded_traffic_people",
    ),
    "4_random_youtube_driving": save_control_points_artifact(
        control_points_4_youtube_video_path,
        control_points_4_youtube_data,
        "4__control_points__3m_drive_suburbs",
    ),
    "5_hard_vehicle_crowd": save_control_points_artifact(
        control_points_5_hard_video_path,
        control_points_5_hard_data,
        "5__control_points__3m_vehicle_crowd_hard",
    ),
}

control_points_artifacts

control_points_background_artifacts = {
    "1_lunar_lander": save_control_points_background_artifact(
        control_points_1_lunar_background_data,
        "1__control_points__lunar_lander",
    ),
    "2_car_racing": save_control_points_background_artifact(
        control_points_2_car_background_data,
        "2__control_points__car_racing_solved_full",
    ),
    "3_recorded_traffic_people": save_control_points_background_artifact(
        control_points_3_traffic_background_data,
        "3__control_points__recorded_traffic_people",
    ),
    "4_random_youtube_driving": save_control_points_background_artifact(
        control_points_4_youtube_background_data,
        "4__control_points__3m_drive_suburbs",
    ),
    "5_hard_vehicle_crowd": save_control_points_background_artifact(
        control_points_5_hard_background_data,
        "5__control_points__3m_vehicle_crowd_hard",
    ),
}

control_points_background_artifacts


# Data Review

In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

CONTROL_POINTS_DIR = Path("artifacts/research/control_points")
BACKGROUND_POINTS_DIR = Path("artifacts/research/background_points")

def load_control_points_pair(shortcut, stem):
    cp_path = CONTROL_POINTS_DIR / f"{stem}.csv"
    bckg_path = BACKGROUND_POINTS_DIR / f"{stem}__background_points.csv"
    cp_df = pd.read_csv(cp_path)
    bckg_df = pd.read_csv(bckg_path)
    print(f"{shortcut}_cp_df:", cp_df.shape, cp_path)
    print(f"{shortcut}_bckg_df:", bckg_df.shape, bckg_path)
    display(cp_df.head())
    display(bckg_df.head())
    return cp_df, bckg_df


## 1 - Lunar Lander CSV Review


In [ ]:
lunar_cp_df, lunar_bckg_df = load_control_points_pair("lunar", "1__control_points__lunar_lander")


In [ ]:
lunar_bckg_df.shape

## 2 - Car Racing CSV Review


In [ ]:
car_cp_df, car_bckg_df = load_control_points_pair("car", "2__control_points__car_racing_solved_full")


## 3 - Recorded Traffic With People CSV Review


In [ ]:
traffic_cp_df, traffic_bckg_df = load_control_points_pair("traffic", "3__control_points__recorded_traffic_people")


## 4 - Random YouTube Driving Scene CSV Review


In [ ]:
youtube_cp_df, youtube_bckg_df = load_control_points_pair("youtube", "4__control_points__3m_drive_suburbs")


## 5 - Hard Vehicle-Crowd Interaction CSV Review


In [ ]:
hard_cp_df, hard_bckg_df = load_control_points_pair("hard", "5__control_points__3m_vehicle_crowd_hard")


# Action Encoder


In [ ]:
lunar_cp_df[lunar_cp_df['track_age']==40]

In [ ]:
temp_df = lunar_bckg_df[(lunar_bckg_df['frame']==40) & (lunar_bckg_df['valid']==1)][['frame','background_point_id','x','y','valid','dx','dy','speed']]
temp_df

In [ ]:
import matplotlib.pyplot as plt

plt.plot(temp_df['x'], temp_df['y'], 'o', markersize=3, color='blue')

## Define Variables

### Control Points

$$P_{i,t} =\begin{bmatrix}p_{i,t}^{(1)} \\p_{i,t}^{(2)} \\\vdots \\p_{i,t}^{(K)}\end{bmatrix}=
\begin{bmatrix}
x_{i,t}^{(1)} & y_{i,t}^{(1)} \\
x_{i,t}^{(2)} & y_{i,t}^{(2)} \\
\vdots & \vdots \\
x_{i,t}^{(K)} & y_{i,t}^{(K)}
\end{bmatrix}
\in \mathbb{R}^{K \times 2}$$

where:

$P_{i,t}$ - all $K$ control points of segmentation blob/object $i$ in frame $t$, including the centroid and selected boundary/control points.

$i$ - object/track index, corresponding to `track_id`.

$t$ - frame index.

$p_{i,t}^{(k)} = [x_{i,t}^{(k)}, y_{i,t}^{(k)}]$ - the $k$-th control point of object $i$ in frame $t$.

In [ ]:
lunar_cp_df

In [ ]:
CONTROL_POINT_NAMES = [
    "centroid",
    "axis_front",
    "axis_back",
    "axis_left",
    "axis_right",
    "contact_low",
]

lunar_cp_df_reduced = lunar_cp_df[
    lunar_cp_df["point_name"].isin(CONTROL_POINT_NAMES)
][
    [
        "frame",
        "track_id",
        "point_name",
        "x",
        "y",
        "box_x1",
        "box_y1",
        "box_x2",
        "box_y2",
    ]
].copy()

lunar_cp_df_reduced["point_name"] = pd.Categorical(
    lunar_cp_df_reduced["point_name"],
    categories=CONTROL_POINT_NAMES,
    ordered=True,
)

lunar_cp_df_reduced = lunar_cp_df_reduced.sort_values(
    ["track_id", "frame", "point_name"]
).reset_index(drop=True)

Each blob has frame, track_id, point_name and box metadata which can be used later on for normalization purposes.

### Background

$$B_t =\begin{bmatrix}b_t^{(1)} \\b_t^{(2)} \\\vdots \\b_t^{(N)}\end{bmatrix}=\begin{bmatrix}
x_{bckg,t}^{(1)} & y_{bckg,t}^{(1)} & dx_{bckg,t}^{(1)} & dy_{bckg,t}^{(1)} \\
x_{bckg,t}^{(2)} & y_{bckg,t}^{(2)} & dx_{bckg,t}^{(2)} & dy_{bckg,t}^{(2)} \\
\vdots & \vdots & \vdots & \vdots \\
x_{bckg,t}^{(N)} & y_{bckg,t}^{(N)} & dx_{bckg,t}^{(N)} & dy_{bckg,t}^{(N)}
\end{bmatrix}
\in \mathbb{R}^{N \times 4}
$$

where:

$B_t$ - selected background optical-flow points in frame $t$.

$N$ - fixed number of selected background points per frame.

$b_t^{(j)} = [x_{bckg,t}^{(j)}, y_{bckg,t}^{(j)}, dx_{bckg,t}^{(j)}, dy_{bckg,t}^{(j)}]$ - the $j$-th selected background point in frame $t$.

$x_{bckg,t}^{(j)}, y_{bckg,t}^{(j)}$ - image coordinates of the $j$-th selected background point in frame $t$.

$dx_{bckg,t}^{(j)}, dy_{bckg,t}^{(j)}$ - optical-flow displacement of the $j$-th selected background point from frame $t-1$ to frame $t$.

Additionally, the scalar optical-flow speed of each selected background point is:

$$s_{bckg,t}^{(j)} = \sqrt{\left(dx_{bckg,t}^{(j)}\right)^2 + \left(dy_{bckg,t}^{(j)}\right)^2}$$

where:

$s_{bckg,t}^{(j)}$ - magnitude of the optical-flow displacement of the $j$-th selected background point from frame $t-1$ to frame $t$.

In [ ]:
N_BACKGROUND_POINTS = 100

lunar_bckg_df_reduced = lunar_bckg_df[
    [
        "frame",
        "background_point_id",
        "x",
        "y",
        "dx",
        "dy",
        "speed",
        "valid",
        "prev_valid",
    ]
].copy()

lunar_bckg_df_selected = (
    lunar_bckg_df_reduced
    .sort_values(["frame", "background_point_id"])
    .groupby("frame", group_keys=False)
    .apply(
        lambda frame_df: frame_df.sample(
            n=min(N_BACKGROUND_POINTS, len(frame_df)),
            random_state=42,
        )
    )
    .sort_values(["frame", "background_point_id"])
    .reset_index(drop=True)
)

In [ ]:
lunar_bckg_df_selected

In [ ]:
lunar_bckg_df_selected.groupby("frame")["background_point_id"].count().describe()

## Action Embedding

In this step, we define motion \(M\), which represents the relation between the control-point displacement of a segmentation blob and the background-point displacement.

$$ A^x_{i,t} = \underbrace{\begin{bmatrix} x_{cp,i,t}^{(1)} \\ x_{cp,i,t}^{(2)} \\ \vdots \\ x_{cp,i,t}^{(K)} \end{bmatrix}}_{X_{cp,i,t} \in \mathbb{R}^{K \times 1}} \; \underbrace{\begin{bmatrix} x_{bckg,t}^{(1)} \\ x_{bckg,t}^{(2)} \\ \vdots \\ x_{bckg,t}^{(N)} \end{bmatrix}^{T}}_{X_{bckg,t}^{T} \in \mathbb{R}^{1 \times N}} $$

$$ A^y_{i,t} = \underbrace{\begin{bmatrix} y_{cp,i,t}^{(1)} \\ y_{cp,i,t}^{(2)} \\ \vdots \\ y_{cp,i,t}^{(K)} \end{bmatrix}}_{Y_{cp,i,t} \in \mathbb{R}^{K \times 1}} \; \underbrace{\begin{bmatrix} y_{bckg,t}^{(1)} \\ y_{bckg,t}^{(2)} \\ \vdots \\ y_{bckg,t}^{(N)} \end{bmatrix}^{T}}_{Y_{bckg,t}^{T} \in \mathbb{R}^{1 \times N}} $$

This is an outer product, not a dot product. A column vector multiplied by a transposed column vector produces a matrix.

$$ A^x_{i,t} \in \mathbb{R}^{K \times N}, \quad A^y_{i,t} \in \mathbb{R}^{K \times N} $$

Each entry stores one multiplicative relation between one control point and one selected background point.

$$ A^x_{i,t}(k,j) = x_{cp,i,t}^{(k)} x_{bckg,t}^{(j)} $$

$$ A^y_{i,t}(k,j) = y_{cp,i,t}^{(k)} y_{bckg,t}^{(j)} $$

The full action embedding stacks the x-relation matrix and the y-relation matrix.

$$ A_{i,t} = \left(A^x_{i,t}, A^y_{i,t}\right) \in \mathbb{R}^{K \times N \times 2} $$

Before using the action embedding as a model input, the embedding can be normalized.

For min-max normalization:

$$ A^{norm}_{i,t} = \frac{A_{i,t} - \min(A_{i,t})}{\max(A_{i,t}) - \min(A_{i,t}) + \epsilon} $$

The normalized flattened token is:

$$ z_{i,t} = \operatorname{flatten}(A^{norm}_{i,t}) $$

where:

$z_{i,t}$ - normalized action embedding token for object $i$ in frame $t$.

$\epsilon$ - small constant for numerical stability.

### Sample

#### Plot 10 Consecutive Frames

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# select 10 consecutive frames for visualization
start_frame = np.random.randint(
    lunar_bckg_df_selected["frame"].min(),
    lunar_bckg_df_selected["frame"].max() - 9,
)
frames = list(range(start_frame, start_frame + 10))

fig, ax = plt.subplots(10, 1, figsize=(8, 40))

for row_idx, frame in enumerate(frames):
    frame_tracks = lunar_cp_df_reduced[
        lunar_cp_df_reduced["frame"] == frame
    ]["track_id"]

    if frame_tracks.empty:
        ax[row_idx].set_title(f"frame={frame} | no control points")
        ax[row_idx].axis("off")
        continue

    track_id = frame_tracks.iloc[0]

    frame_cp = lunar_cp_df_reduced[
        (lunar_cp_df_reduced["frame"] == frame)
        & (lunar_cp_df_reduced["track_id"] == track_id)
    ].copy()

    frame_bckg = lunar_bckg_df_selected[
        lunar_bckg_df_selected["frame"] == frame
    ].copy()

    ax[row_idx].scatter(
        frame_bckg["x"],
        frame_bckg["y"],
        s=12,
        alpha=0.35,
        label="selected background points",
    )

    ax[row_idx].scatter(
        frame_cp["x"],
        frame_cp["y"],
        s=90,
        c="red",
        label="control points",
    )

    for _, cp_row in frame_cp.iterrows():
        ax[row_idx].text(
            cp_row["x"] + 3,
            cp_row["y"] + 3,
            cp_row["point_name"],
            fontsize=8,
            color="red",
        )

    ax[row_idx].invert_yaxis()
    ax[row_idx].set_xlabel("x image coordinate")
    ax[row_idx].set_ylabel("y image coordinate")
    ax[row_idx].set_title(f"frame={frame}, track_id={track_id}")

ax[0].legend(loc="upper right")
plt.suptitle(
    f"Control Points and Selected Background Points Before Outer Product | "
    f"frames {start_frame}-{start_frame + 9}",
    fontsize=16,
    y=1.002,
)
plt.tight_layout()
plt.show()

#### Single Action Embedding

In [ ]:
x_cp = frame_cp["x"]
y_cp = frame_cp["y"]

x_bckg = frame_bckg["x"]
y_bckg = frame_bckg["y"]

X_cp = x_cp.values.reshape(-1, 1)
Y_cp = y_cp.values.reshape(-1, 1)

X_bckg = x_bckg.values.reshape(-1, 1)
Y_bckg = y_bckg.values.reshape(-1, 1)

A_x = X_cp @ X_bckg.T
A_y = Y_cp @ Y_bckg.T

A = np.stack([A_x, A_y], axis=-1)

fig, ax = plt.subplots(1, 2, figsize=(12, 5))

im0 = ax[0].imshow(A_x, cmap="viridis", aspect="auto")
ax[0].set_title(f"$A^x$ | frame={frame}, track_id={track_id}")
ax[0].set_xlabel("selected background point index")
ax[0].set_ylabel("control point index")
fig.colorbar(im0, ax=ax[0])

im1 = ax[1].imshow(A_y, cmap="viridis", aspect="auto")
ax[1].set_title(f"$A^y$ | frame={frame}, track_id={track_id}")
ax[1].set_xlabel("selected background point index")
ax[1].set_ylabel("control point index")
fig.colorbar(im1, ax=ax[1])

plt.suptitle(
    f"Action Embedding Outer Product | frame={frame}, track_id={track_id} | A shape={A.shape}",
    fontsize=16,
    y=1.02,
)
plt.tight_layout()
plt.show()

In [ ]:
A_x.shape, A_y.shape

#### Normalize Single Action Embedding

In [ ]:
A_x_norm = (A_x - A_x.mean()) / (A_x.std() + 1e-8)
A_y_norm = (A_y - A_y.mean()) / (A_y.std() + 1e-8)

A_norm = np.stack([A_x_norm, A_y_norm], axis=-1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 5))

im0 = ax[0].imshow(A_x_norm, cmap="viridis", aspect="auto")
ax[0].set_title(f"Normalized $A^x$ | frame={frame}, track_id={track_id}")
ax[0].set_xlabel("selected background point index")
ax[0].set_ylabel("control point index")
fig.colorbar(im0, ax=ax[0])

im1 = ax[1].imshow(A_y_norm, cmap="viridis", aspect="auto")
ax[1].set_title(f"Normalized $A^y$ | frame={frame}, track_id={track_id}")
ax[1].set_xlabel("selected background point index")
ax[1].set_ylabel("control point index")
fig.colorbar(im1, ax=ax[1])

plt.suptitle(
    f"Normalized Action Embedding Outer Product | frame={frame}, track_id={track_id} | A_norm shape={A_norm.shape}",
    fontsize=16,
    y=1.02,
)
plt.tight_layout()
plt.show()

### Full Action Embeddings with Normalization for 1 Video

Now, lets create full sequence of Action Embeddings

In [ ]:
lunar_action_embeddings = {}

common_frames = sorted(
    set(lunar_cp_df_reduced["frame"].unique())
    & set(lunar_bckg_df_selected["frame"].unique())
)

for frame in common_frames:
    frame_bckg = lunar_bckg_df_selected[
        lunar_bckg_df_selected["frame"] == frame
    ].sort_values("background_point_id")

    X_bckg = frame_bckg[["x"]].to_numpy()
    Y_bckg = frame_bckg[["y"]].to_numpy()

    frame_embeddings = {}

    for track_id, frame_cp in lunar_cp_df_reduced[
        lunar_cp_df_reduced["frame"] == frame
    ].groupby("track_id", sort=False):

        frame_cp = frame_cp.sort_values("point_name")

        X_cp = frame_cp[["x"]].to_numpy()
        Y_cp = frame_cp[["y"]].to_numpy()

        A_x = X_cp @ X_bckg.T
        A_y = Y_cp @ Y_bckg.T
        A = np.stack([A_x, A_y], axis=-1)

        A_min = A.min()
        A_max = A.max()
        A_norm = (A - A_min) / (A_max - A_min + 1e-8)

        A_x_norm = A_norm[:, :, 0]
        A_y_norm = A_norm[:, :, 1]

        frame_embeddings[track_id] = {
            "A_x": A_x,
            "A_y": A_y,
            "A": A,
            "A_min": A_min,
            "A_max": A_max,
            "A_x_norm": A_x_norm,
            "A_y_norm": A_y_norm,
            "A_norm": A_norm,
            "token_flat": A.reshape(-1),
            "token_norm_flat": A_norm.reshape(-1),
            "control_point_dim": len(frame_cp),
            "background_dim": len(frame_bckg),
            "relation_channels": A.shape[-1],
            "flattened_dim": A.reshape(-1).shape[0],
        }

    lunar_action_embeddings[frame] = frame_embeddings

In [ ]:
for frame, tracks in lunar_action_embeddings.items():
    for track_id, emb in tracks.items():
        print(
            f"frame={frame:>3} "
            f"track_id={track_id:>3} "
            f"A_shape={str(emb['A'].shape):>14} "
            f"A_norm_shape={str(emb['A_norm'].shape):>14} "
            f"background_dim={emb['background_dim']:>4} "
            f"control_point_dim={emb['control_point_dim']:>2} "
            f"relation_channels={emb['relation_channels']:>2} "
            f"flattened_dim={emb['flattened_dim']:>5} "
            f"A_min={emb['A_min']:>12.4f} "
            f"A_max={emb['A_max']:>12.4f} "
            f"A_norm_min={emb['A_norm'].min():>7.4f} "
            f"A_norm_max={emb['A_norm'].max():>7.4f}"
        )

#### Change of Action Embedding

In [ ]:
track_id = 1

track_frames = [
    frame for frame in sorted(lunar_action_embeddings)
    if track_id in lunar_action_embeddings[frame]
]

track_tokens = np.stack([
    lunar_action_embeddings[frame][track_id]["token_norm_flat"]
    for frame in track_frames
])

delta_norm = np.linalg.norm(np.diff(track_tokens, axis=0), axis=1)

plt.plot(track_frames[1:], delta_norm)
plt.xlabel("frame")
plt.ylabel("||A_norm_t - A_norm_t-1||")
plt.title(f"Normalized action embedding temporal change | track_id={track_id}")
plt.show()

This plot shows:

$$
\|A^{norm}_{t} - A^{norm}_{t-1}\|
$$

for `track_id=1`.

It measures how much the normalized action embedding changes between consecutive frames.

Interpretation:

- lower values mean the object-background relation changes less between frames
- higher values mean the object-background relation changes more strongly
- peaks indicate frames where the encoded action changes sharply
- the curve is not flat, so the embedding carries temporal signal

The y-axis is larger than 1 because it is the L2 norm of a high-dimensional vector, not a single normalized scalar.

# Transformer Model

### Transformer Hyperparameters


In [ ]:
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

device = "cuda" if torch.cuda.is_available() else "cpu"

torch.manual_seed(42)
np.random.seed(42)

sample_frame = next(iter(lunar_action_embeddings))
sample_track_id = next(iter(lunar_action_embeddings[sample_frame]))
sample_embedding = lunar_action_embeddings[sample_frame][sample_track_id]

control_point_dim = sample_embedding["control_point_dim"]
background_dim = sample_embedding["background_dim"]
relation_channels = sample_embedding["relation_channels"]
flattened_dim = sample_embedding["flattened_dim"]

seq_len = 4
batch_size = 8
embed_dim = 256
n_heads = 4
n_blocks = 4
dropout = 0.1

assert embed_dim % n_heads == 0

{
    "device": device,
    "control_point_dim": control_point_dim,
    "background_dim": background_dim,
    "relation_channels": relation_channels,
    "flattened_dim": flattened_dim,
    "seq_len": seq_len,
    "batch_size": batch_size,
    "embed_dim": embed_dim,
    "n_heads": n_heads,
    "n_blocks": n_blocks,
    "dropout": dropout,
}


The transformer operates on a sequence of normalized action embedding tokens.

For object $i$, each frame $t$ has a normalized action embedding token:

$$ z_{i,t} = \operatorname{flatten}(A^{norm}_{i,t}) $$

The input sequence is:

$$ X_i = [z_{i,1}, z_{i,2}, \dots, z_{i,T-1}] $$

The target sequence is the same sequence shifted by one frame:

$$ Y_i = [z_{i,2}, z_{i,3}, \dots, z_{i,T}] $$

The transformer learns to predict the next action embedding token from previous action embedding tokens:

$$ \hat{Y}_i = f_{\theta}(X_i) $$

The training loss is mean squared error between the predicted next action embedding sequence and the true next action embedding sequence:

$$ \mathcal{L} = \frac{1}{T-1} \sum_{t=1}^{T-1} \left\| \hat{z}_{i,t+1} - z_{i,t+1} \right\|_2^2 $$

This is GPT-style causal sequence modeling, but over continuous action embedding vectors instead of discrete text tokens.

### Action Embedding Token Sequence

Prepare and visualize the normalized action embedding token sequence for one tracked object.


In [ ]:
track_id = 1

track_frames = [
    frame for frame in sorted(lunar_action_embeddings)
    if track_id in lunar_action_embeddings[frame]
]

track_tokens = np.stack([
    lunar_action_embeddings[frame][track_id]["token_norm_flat"]
    for frame in track_frames
])

plt.imshow(track_tokens, aspect="auto", cmap="viridis")

### Sequence Dataset

Build causal training windows from normalized action embedding tokens.


In [ ]:
track_id = 1

track_frames = [
    frame for frame in sorted(lunar_action_embeddings)
    if track_id in lunar_action_embeddings[frame]
]

track_tokens = np.stack([
    lunar_action_embeddings[frame][track_id]["A_norm"].reshape(-1)
    for frame in track_frames
]).astype(np.float32)

split_idx = int(len(track_tokens) * 0.8)
train_tokens = track_tokens[:split_idx]
test_tokens = track_tokens[split_idx - seq_len - 1:]


def get_data_batch(train=True, batch_size=batch_size):
    data = train_tokens if train else test_tokens
    max_start = len(data) - seq_len - 1

    if max_start < 0:
        raise ValueError(f"Not enough tokens: len(data)={len(data)}, seq_len={seq_len}")

    starts = np.random.randint(0, max_start + 1, size=batch_size)

    X = np.stack([data[start:start + seq_len] for start in starts])
    y = np.stack([data[start + 1:start + seq_len + 1] for start in starts])

    return (
        torch.tensor(X, dtype=torch.float32),
        torch.tensor(y, dtype=torch.float32),
    )

len(track_tokens), len(train_tokens), len(test_tokens), track_tokens.shape


### Attention and Transformer Block

Define the causal multi-head attention block and the transformer block used by the action embedding model.


In [ ]:
class MultiHeadAttention(nn.Module):
  def __init__(self):
    super().__init__()

    # number of attention heads
    self.num_heads = n_heads
    self.head_dim = embed_dim // n_heads

    # Q, K, V
    self.QKV = nn.Linear(embed_dim, 3*embed_dim, bias = True)

    # linear mixing after attention
    self.W0 = nn.Linear(embed_dim, embed_dim, bias = True)

    # n dropout not defined here b/c it's in F.scaled_dot_product_attention

  def forward(self, x):

    # sizes for later use
    B, T, E = x.shape # [batch, seq_len, embed_dim]

    # push data through QKV in one matrix
    qkv = self.QKV(x)
    q,k,v = torch.split(qkv,E,dim=2)

    # reshape to [B,T,nHeads, head_dim]

    q = q.view(B,T,self.num_heads, self.head_dim).transpose(1,2)
    k = k.view(B,T,self.num_heads, self.head_dim).transpose(1,2)
    v = v.view(B,T,self.num_heads, self.head_dim).transpose(1,2)

    dropp = dropout if self.training==True else 0
    out = F.scaled_dot_product_attention(q,k,v,is_causal=True,dropout_p=dropp)

    # recombine heads: (B, n_heads, T, head_dim) -> [B, T, E]
    # out = out.transpose(1,2).view(B,T,E)
    out = out.transpose(1, 2).contiguous().view(B, T, E)

    # finally, linearly mix the attention heads
    out = self.W0(out)

    return out

class TransformerBlock(nn.Module):
  def __init__(self):
    super().__init__()

    # attention subblock
    self.layernorm_1 = nn.LayerNorm(embed_dim, eps = 1e-5)
    self.attn = MultiHeadAttention()

    # linear feedforward (MLP) subblock
    self.layernorm_2 = nn.LayerNorm(embed_dim, eps = 1e-5)
    # 4x expansion then back
    self.mlp_1 = nn.Linear(embed_dim, 4*embed_dim, bias = True)
    self.gelu  = nn.GELU()
    self.mlp_2 = nn.Linear(4*embed_dim, embed_dim, bias = True)

    # n transformer block dropout
    self.trn_dropout = nn.Dropout(dropout)

  def forward(self, x):

    # attention
    x_att = self.layernorm_1(x) # pre-attention norm
    x_att = x + self.trn_dropout(self.attn(x_att)) # n attention -> dropout -> add

    # MLP
    x_ff = self.layernorm_2(x_att) # pre-MLP normalizaiton
    x_ff = self.mlp_2(self.gelu(self.mlp_1(x_ff)))
    x_ff = x_att + self.trn_dropout(x_ff)

    return x_ff

### Transformer Model Wrapper

Model wrapper adapted from the Cohen language-model baseline. This subsection is the place to replace discrete token embedding with continuous action embedding projection.


In [ ]:
class ActionEmbeddingModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.input_proj = nn.Linear(flattened_dim, embed_dim, bias=True)
        self.wpe = nn.Embedding(seq_len, embed_dim)
        self.emb_dropout = nn.Dropout(dropout)

        self.transformerBlocks = nn.Sequential(
            *[TransformerBlock() for _ in range(n_blocks)]
        )

        self.layernorm_final = nn.LayerNorm(embed_dim, eps=1e-5)
        self.final_head = nn.Linear(embed_dim, flattened_dim, bias=True)

        self.apply(self.weightInits)

    def weightInits(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0, std=.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)

        if isinstance(module, nn.Embedding):
            nn.init.xavier_normal_(module.weight)

    def forward(self, z):
        # z: [B, T, flattened_dim]
        token_emb = self.input_proj(z)
        posit_emb = self.wpe(torch.arange(z.shape[1], device=z.device))

        x = token_emb + posit_emb
        x = self.emb_dropout(x)
        x = self.transformerBlocks(x)
        x = self.layernorm_final(x)
        out = self.final_head(x)

        return out


### Model Initialization


In [ ]:
model = ActionEmbeddingModel().to(device)


### Loss and Optimizer


In [ ]:
loss_function = nn.MSELoss().to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=0.01)

X, y = get_data_batch()
pred = model(X.to(device))
loss = loss_function(pred[:, -1, :], y[:, -1, :].to(device))
loss

### Training Loop


In [ ]:
num_samples = 100

train_loss = []
test_loss = []
test_steps = []

for sampli in range(num_samples):
    X, y = get_data_batch()

    model.zero_grad(set_to_none=True)

    pred = model(X.to(device))
    loss = loss_function(pred, y.to(device))

    loss.backward()
    optimizer.step()

    train_loss.append(loss.item())

    with torch.no_grad():
        model.eval()

        X_test, y_test = get_data_batch(False)
        pred_test = model(X_test.to(device))
        test_loss_value = loss_function(pred_test, y_test.to(device))

        test_loss.append(test_loss_value.item())
        test_steps.append(sampli)

        model.train()

    print(
        f"Sample {sampli:4}, "
        f"train loss: {train_loss[-1]:8.5f}, "
        f"test loss: {test_loss[-1]:8.5f}"
    )

## Losses

In [ ]:
plt.figure(figsize=(10, 4))

plt.plot(range(len(train_loss)), train_loss, label="train loss")
plt.plot(range(len(test_loss)), test_loss, marker="o", label="test loss")

plt.xlabel("training step")
plt.ylabel("MSE loss")
plt.title("Action Embedding Transformer Loss")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## Action Embeddings Evaluation

In [ ]:
with torch.no_grad():
    model.eval()

    X, y = get_data_batch(False)
    pred = model(X.to(device)).cpu()

    model.train()

sample_idx = 0

true_embedding = y[sample_idx, -1].numpy()
pred_embedding = pred[sample_idx, -1].numpy()

fig, ax = plt.subplots(1, 2, figsize=(12, 4))

ax[0].plot(true_embedding, label="true")
ax[0].plot(pred_embedding, label="pred", alpha=0.75)
ax[0].set_title("Flattened Action Embedding: True vs Pred")
ax[0].set_xlabel("flattened embedding index")
ax[0].set_ylabel("value")
ax[0].legend()

ax[1].scatter(true_embedding, pred_embedding, s=8, alpha=0.5)
ax[1].set_title("Predicted vs True Embedding Values")
ax[1].set_xlabel("true")
ax[1].set_ylabel("pred")

plt.tight_layout()
plt.show()